# SAM 3 + SpatialTrackerV2 → object-centric 4D environment (Colab T4)

این نوت‌بوک یک ویدئو را به محیط شیءمحورِ قابل‌استفاده در schema ضمیمه‌شده تبدیل می‌کند:

1. **SAM 3 Tracker** روی چند فریم در سراسر ویدئو، با automatic mask generation و بدون vocabulary ثابت، ناحیه‌های object-like را پیدا می‌کند.
2. از داخل هر mask، point query با زمان تولد واقعی `(t, x, y)` ساخته می‌شود.
3. **SpatialTrackerV2 Online** عمق، دوربین و مسیر سه‌بعدی نقطه‌ها را برآورد می‌کند.
4. fragmentهای دیده‌شده در زمان‌های مختلف با شواهد point، appearance، 3D و class به یک `world_track_id` وصل می‌شوند.
5. یک OBB سه‌بعدیِ پایدار و فیلتر حرکت ساخته می‌شود؛ هنگام occlusion/out-of-view باکس حذف نمی‌شود و به‌صورت `predicted` ادامه می‌یابد.

خروجی‌ها:

- `scene.environment.json`: schemaVersion `1.0` با وضعیت صریح کالیبراسیون در `coordinates.metricScale`
- `scene.tracking.json`: وضعیت `observed / occluded / out_of_view / lost`، confidence، uncertainty، lineage و دوربین اصلی
- `raw_spatial_tracks.npz`: خروجی خام قابل‌پردازش
- `trajectory_preview.png` و `tracking_preview.mp4`
- یک ZIP شامل همه‌ی موارد بالا

## قبل از Run all

- در Colab از **Runtime → Change runtime type → T4 GPU** استفاده کنید.
- دسترسی checkpoint را در [facebook/sam3](https://huggingface.co/facebook/sam3) بپذیرید و token دارای مجوز read آماده کنید.
- کد tracker از [SpaTrackerV2 GitHub](https://github.com/henry123-boy/SpaTrackerV2) و وزن‌ها از [Online](https://huggingface.co/Yuxihenry/SpatialTrackerV2-Online) و [Front](https://huggingface.co/Yuxihenry/SpatialTrackerV2_Front) دریافت می‌شوند.
- این profile برای **یک شات پیوسته و حداکثر ۱۰ ثانیه** است. `MAX_TRACK_FRAMES=10` کل کلیپ را sparse sample می‌کند تا روی T4 عملی بماند؛ ویدئوی بلندتر را به کلیپ‌های overlapدار تقسیم کنید.
- برای بودجهٔ T4، حداکثر ۸ mask برترِ هر فریم نمونه‌برداری‌شده seed می‌شوند. صحنهٔ شلوغ‌تر به چند پاس یا GPU بزرگ‌تر نیاز دارد.

> محدودیت‌های بنیادی: automatic mask generation می‌تواند part/stuff را هم object حساب کند یا شیء بسیار کوچک را جا بیندازد؛ تضمین منطقیِ «تمام اشیای فیزیکی» وجود ندارد. ویدیوی RGB تک‌دوربین نیز scale مطلقِ متر را تعیین نمی‌کند؛ برای خروجی metric باید `METERS_PER_TRACKER_UNIT` را با مرجع واقعی کالیبره کنید.


In [ ]:
# 0. Configuration — the defaults are intentionally conservative for a 16 GB T4.
import json
import math
from pathlib import Path

INPUT_VIDEO = None  # Example: "/content/my_video.mp4"; None opens Colab upload.
ENVIRONMENT_ID = "video-environment-001"

TARGET_FPS = 2.0
MAX_TRACK_FRAMES = 10          # covers the complete clip with sparse sampling
MAX_CLIP_DURATION_SECONDS = 10.0  # split longer videos into continuous shots
PROCESSED_SIZE = 518           # shared SAM/SpatialTracker pixel coordinates
DISCOVERY_EVERY = 1            # detect entry on every sampled frame (slower, higher recall)
ABORT_ON_SUSPECTED_SCENE_CUT = True

# SAM 3 Tracker automatic-mask generation (class-agnostic).
# More points improve small-object recall but take longer. Batch size controls VRAM.
SAM3_POINTS_PER_BATCH = 4
SAM3_POINTS_PER_CROP = 16
SAM3_PRED_IOU_THRESHOLD = 0.78
SAM3_STABILITY_THRESHOLD = 0.86
SAM3_MIN_AREA_FRACTION = 0.0004
MAX_OBJECTS_PER_DISCOVERY = 8  # T4 profile: top object-like masks per sampled frame

POINTS_PER_OBJECT = 4
# Aligned with the discovery cap so no accepted mask is silently starved.
MAX_TOTAL_QUERIES = MAX_TRACK_FRAMES * MAX_OBJECTS_PER_DISCOVERY * POINTS_PER_OBJECT
SUPPORT_POINTS = 64
TRACK_ITERATIONS = 2
TRACK_CONFIDENCE = 0.35
VISIBILITY_THRESHOLD = 0.95     # SpaTracker visibility is sigmoid(logit)+0.2
RANDOM_SEED = 17

ASSOCIATION_THRESHOLD = 0.50
MAX_REID_GAP_SECONDS = 10.0       # keep identities eligible for the whole allowed clip
MAX_PREDICTION_SECONDS = 4.0
MAX_MEDIAN_REPROJECTION_ERROR_PX = 8.0
MAX_P95_REPROJECTION_ERROR_PX = 30.0

# None means relative scene scale. Set a measured calibration factor for true meters.
# Example: if a known 2 m span measures 1.6 tracker units, use 2/1.6 = 1.25.
METERS_PER_TRACKER_UNIT = None

if PROCESSED_SIZE != 518:
    raise ValueError("SpatialTrackerV2 Front requires PROCESSED_SIZE=518 in this notebook.")
if not math.isfinite(TARGET_FPS) or TARGET_FPS <= 0:
    raise ValueError("TARGET_FPS must be finite and positive.")
if not 4 <= MAX_TRACK_FRAMES <= 20:
    raise ValueError("Use 4..20 frames; split longer/dense jobs into overlapping shots.")
if DISCOVERY_EVERY < 1:
    raise ValueError("DISCOVERY_EVERY must be at least 1.")
if POINTS_PER_OBJECT < 3:
    raise ValueError("At least three object seeds are required for depth validation.")

WORK_ROOT = Path("/content/4d_env")
FRAME_ROOT = WORK_ROOT / "prepared"
SAM_ROOT = WORK_ROOT / "sam3_discovery"
QUERY_PATH = WORK_ROOT / "object_queries.npz"
TRACK_PATH = WORK_ROOT / "spatial_tracks.npz"
OUTPUT_ROOT = WORK_ROOT / "environment_bundle"

SAM3_MODEL_ID = "facebook/sam3"
SAM3_MODEL_REVISION = "3c879f39826c281e95690f02c7821c4de09afae7"
TRANSFORMERS_REVISION = "9eb65c3231fa04dc746c73dfcf5777be9c31d9d4"
SAM3_REVISION = f"{SAM3_MODEL_ID}@{SAM3_MODEL_REVISION};transformers@{TRANSFORMERS_REVISION}"
SPATRACKER_REVISION = "7e12274c52077860cebfe007a6290777db43b63c"
SPATRACKER_FRONT_REVISION = "2e7164a93770cb934861d7de579bb81f58ad0b03"
SPATRACKER_ONLINE_REVISION = "d1e3b89e0ab10b604b0caa1cd001d8c24182bd86"


## 1. نصب دو محیط جدا

SAM 3 و SpatialTrackerV2 به stackهای متفاوتی نیاز دارند. بنابراین هر مدل در venv خودش اجرا می‌شود و فقط فایل‌های CPU بین مرحله‌ها ردوبدل می‌شوند. مسیر رسمی Transformers برای SAM3 Tracker در FP16 استفاده می‌شود؛ SpatialTracker نیز در FP16 با attention سازگار T4 اجرا می‌شود.


In [ ]:
import os
import platform
import shutil
import subprocess
import sys

def run(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("+", " ".join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU. Choose Runtime > Change runtime type > T4 GPU.")
gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
    text=True,
).strip()
print("GPU:", gpu_info)
print("Host Python:", sys.version.split()[0], platform.platform())
if "T4" not in gpu_info:
    print("NOTE: this notebook is tuned for T4; another CUDA GPU may need different settings.")
free_disk = shutil.disk_usage("/content").free / 1024**3
print(f"Free disk: {free_disk:.1f} GiB")
if free_disk < 18:
    raise RuntimeError("At least 18 GiB free disk is recommended for two runtimes and gated weights.")

os.environ["HF_HOME"] = "/content/huggingface_cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_MODULE_LOADING"] = "LAZY"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
(WORK_ROOT / "scripts").mkdir(exist_ok=True)

run([sys.executable, "-m", "pip", "install", "-q", "uv", "opencv-python-headless<5", "scipy", "matplotlib"])
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv installation failed")
run([UV, "python", "install", "3.12", "3.11"])

SPA_REPO = Path("/content/SpaTrackerV2")
if not (SPA_REPO / ".git").exists():
    run(["git", "clone", "--filter=blob:none", "https://github.com/henry123-boy/SpaTrackerV2.git", SPA_REPO])
current = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SPA_REPO, text=True).strip()
if current != SPATRACKER_REVISION:
    run(["git", "fetch", "origin", SPATRACKER_REVISION, "--depth", "1"], cwd=SPA_REPO)
    run(["git", "checkout", "--detach", SPATRACKER_REVISION], cwd=SPA_REPO)

SAM_ENV = Path("/content/venvs/sam3")
SPA_ENV = Path("/content/venvs/spatracker")
if not (SAM_ENV / "bin/python").exists():
    run([UV, "venv", "--python", "3.12", SAM_ENV])
if not (SPA_ENV / "bin/python").exists():
    run([UV, "venv", "--python", "3.11", SPA_ENV])
SAM_PY = SAM_ENV / "bin/python"
SPA_PY = SPA_ENV / "bin/python"

# SAM 3 Tracker automatic mask generation from the official Transformers
# implementation. Code is pinned; the resolved gated-model commit is logged.
run([
    UV, "pip", "install", "--python", SAM_PY,
    "torch==2.7.1", "torchvision==0.22.1",
    "--index-url", "https://download.pytorch.org/whl/cu126",
])
run([
    UV, "pip", "install", "--python", SAM_PY,
    f"git+https://github.com/huggingface/transformers.git@{TRANSFORMERS_REVISION}",
    "accelerate", "opencv-python-headless<5", "pillow",
])

# SpatialTrackerV2: pinned tested stack, without Gradio/Ray/legacy SAM UI.
run([
    UV, "pip", "install", "--python", SPA_PY,
    "torch==2.4.1", "torchvision==0.19.1", "torchaudio==2.4.1",
    "--index-url", "https://download.pytorch.org/whl/cu124",
])
spatial_dependencies = [
    "pydantic==2.10.6", "opencv-python-headless<5", "einops", "einx", "decord",
    "easydict", "safetensors", "scikit-learn", "hydra-core", "omegaconf",
    "albumentations", "matplotlib", "mediapy", "scikit-image", "pycolmap==3.11.1",
    "git+https://github.com/EasternJournalist/utils3d.git@d3a577acf0a9ad7e513a1416449a07b6f47d967f",
    "huggingface_hub", "pyceres==2.4", "kornia==0.8.1", "xformers==0.0.28",
    "timm", "rich", "jaxtyping", "transformers", "flow-vis", "moviepy==1.0.3",
]
run([UV, "pip", "install", "--python", SPA_PY, *spatial_dependencies])

# Patch only the audited, pinned SpatialTracker source sites.
attention_path = SPA_REPO / "models/SpaTrackV2/models/tracker3D/co_tracker/utils.py"
attention = attention_path.read_text()
old_attention = "with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=False):"
new_attention = "with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=True, enable_mem_efficient=True):"
if old_attention in attention:
    attention_path.write_text(attention.replace(old_attention, new_attention))
elif new_attention not in attention:
    raise RuntimeError("Pinned SpatialTracker attention patch site was not found.")

# Preserve query cardinality/order. The upstream block silently drops low-depth
# queries and pads by duplicating other queries without returning an ID remap.
spatial_path = SPA_REPO / "models/SpaTrackV2/models/SpaTrack.py"
spatial_source = spatial_path.read_text()
query_filter_start = "                    if (full_point==False) and (overlap_d is None):\n"
query_filter_end = "\n\n                support_pts_q = get_track_points"
patch_marker = "                    # ENV4D: query depth was prevalidated; preserve point-to-object IDs.\n"
if query_filter_start in spatial_source:
    start = spatial_source.index(query_filter_start)
    end = spatial_source.index(query_filter_end, start)
    spatial_source = spatial_source[:start] + patch_marker + "                    pass" + spatial_source[end:]
    spatial_path.write_text(spatial_source)
elif patch_marker not in spatial_source:
    raise RuntimeError("Pinned SpatialTracker query-remap patch site was not found.")

print("Pinned environments and T4 patches are ready.")


## 2. احراز هویت و انتخاب ویدئو

Token چاپ یا داخل فایل ذخیره نمی‌شود. اگر Colab Secret با نام `HF_TOKEN` دارید از همان استفاده می‌شود؛ در غیر این صورت یک ورودی مخفی نمایش داده خواهد شد.


In [ ]:
import getpass
import shutil

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    hf_token = getpass.getpass("Hugging Face read token (after accepting facebook/sam3): ")
if not hf_token:
    raise RuntimeError("A Hugging Face token with accepted SAM 3 access is required.")
os.environ["HF_TOKEN"] = hf_token

if INPUT_VIDEO is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Set INPUT_VIDEO when running outside Colab.") from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one video file.")
    INPUT_VIDEO = str(Path("/content") / next(iter(uploaded)))
INPUT_VIDEO = Path(INPUT_VIDEO)
if not INPUT_VIDEO.exists():
    raise FileNotFoundError(INPUT_VIDEO)
canonical_video = WORK_ROOT / ("input" + INPUT_VIDEO.suffix.lower())
if INPUT_VIDEO.resolve() != canonical_video.resolve():
    shutil.copy2(INPUT_VIDEO, canonical_video)
INPUT_VIDEO = canonical_video
print("Input:", INPUT_VIDEO, f"({INPUT_VIDEO.stat().st_size / 1024**2:.1f} MiB)")


## 3. Sampling کل ویدئو و فضای پیکسلی مشترک

تمام فریم‌های منتخب letterbox می‌شوند؛ بنابراین maskهای SAM و queryهای SpatialTracker دقیقاً یک دستگاه مختصات دارند. اگر فیلم طولانی باشد، ۱۰ فریم به‌طور یکنواخت کل آن را پوشش می‌دهند تا آبجکت دیررس کاملاً نادیده نماند. برای حرکت سریع باید کلیپ را کوتاه‌تر یا frame cap را با GPU قوی‌تر افزایش دهید.


In [ ]:
%%writefile /content/4d_env/scripts/prepare_video.py
#!/usr/bin/env python3
"""Sample an entire video into a shared letterboxed frame space."""

from __future__ import annotations

import argparse
import json
import shutil
import subprocess
from pathlib import Path

import cv2
import numpy as np


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--target-fps", type=float, default=2.0)
    parser.add_argument("--max-frames", type=int, default=10)
    parser.add_argument("--max-duration", type=float, default=None)
    parser.add_argument("--size", type=int, default=518)
    parser.add_argument("--discovery-every", type=int, default=3)
    parser.add_argument("--sam3-revision", required=True)
    parser.add_argument("--spatracker-revision", required=True)
    return parser.parse_args()


def ffprobe_json(path: Path, *arguments: str) -> dict:
    if shutil.which("ffprobe") is None:
        return {}
    command = [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        *arguments, "-of", "json", str(path),
    ]
    try:
        return json.loads(subprocess.check_output(command, text=True, timeout=120))
    except (OSError, subprocess.SubprocessError, json.JSONDecodeError):
        return {}


def probe_duration(path: Path) -> float | None:
    payload = ffprobe_json(path, "-show_entries", "stream=duration:format=duration")
    raw_value = None
    if payload.get("streams"):
        raw_value = payload["streams"][0].get("duration")
    if raw_value is None:
        raw_value = payload.get("format", {}).get("duration")
    try:
        value = float(raw_value)
    except (TypeError, ValueError):
        return None
    return value if np.isfinite(value) and value > 0 else None


def probe_frame_timestamps(path: Path) -> np.ndarray | None:
    payload = ffprobe_json(
        path,
        "-show_frames",
        "-show_entries", "frame=best_effort_timestamp_time",
    )
    frames = payload.get("frames", [])
    if len(frames) < 4 or any("best_effort_timestamp_time" not in frame for frame in frames):
        return None
    try:
        timestamps = np.asarray(
            [float(frame["best_effort_timestamp_time"]) for frame in frames],
            dtype=np.float64,
        )
    except (TypeError, ValueError):
        return None
    timestamps -= timestamps[0]
    if not np.isfinite(timestamps).all() or np.any(np.diff(timestamps) <= 0):
        return None
    return timestamps


def nearest_frame_indices(timestamps: np.ndarray, target_times: np.ndarray) -> np.ndarray:
    right = np.clip(np.searchsorted(timestamps, target_times), 0, len(timestamps) - 1)
    left = np.clip(right - 1, 0, len(timestamps) - 1)
    choose_left = np.abs(timestamps[left] - target_times) <= np.abs(
        timestamps[right] - target_times
    )
    return np.where(choose_left, left, right).astype(np.int64)


def letterbox(frame: np.ndarray, size: int) -> tuple[np.ndarray, dict]:
    height, width = frame.shape[:2]
    scale = min(size / width, size / height)
    new_width = max(1, int(round(width * scale)))
    new_height = max(1, int(round(height * scale)))
    resized = cv2.resize(frame, (new_width, new_height), interpolation=cv2.INTER_AREA)
    pad_left = (size - new_width) // 2
    pad_top = (size - new_height) // 2
    canvas = np.full((size, size, 3), 255, dtype=np.uint8)
    canvas[pad_top : pad_top + new_height, pad_left : pad_left + new_width] = resized
    info = {
        "sourceSize": [width, height], "processedSize": [size, size],
        "scale": float(scale), "padLeft": int(pad_left), "padTop": int(pad_top),
        "contentRectXYXY": [
            int(pad_left), int(pad_top),
            int(pad_left + new_width), int(pad_top + new_height),
        ],
    }
    return canvas, info


def main() -> None:
    args = parse_args()
    if not np.isfinite(args.target_fps) or args.target_fps <= 0:
        raise ValueError("--target-fps must be finite and positive")
    if args.max_frames < 4:
        raise ValueError("--max-frames must be at least 4")
    if args.max_duration is not None and (
        not np.isfinite(args.max_duration) or args.max_duration <= 0
    ):
        raise ValueError("--max-duration must be finite and positive")
    container_duration = probe_duration(args.input)
    if (
        args.max_duration is not None
        and container_duration is not None
        and container_duration > args.max_duration + 1e-3
    ):
        raise RuntimeError(
            f"Video duration {container_duration:.3f}s exceeds the "
            f"{args.max_duration:.3f}s profile. Split it into overlapping shots."
        )
    capture = cv2.VideoCapture(str(args.input))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video: {args.input}")
    source_fps = float(capture.get(cv2.CAP_PROP_FPS))
    reported_source_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_timestamps = probe_frame_timestamps(args.input)
    if frame_timestamps is not None:
        source_count = len(frame_timestamps)
        if source_count < 4:
            raise RuntimeError("At least four source frames are required.")
        timestamp_mode = "ffprobe-best-effort-pts"
        average_fps = (source_count - 1) / max(float(frame_timestamps[-1]), 1e-6)
        source_fps = float(average_fps)
    else:
        if not np.isfinite(source_fps) or source_fps <= 0:
            raise RuntimeError("Video has neither usable frame PTS nor FPS metadata.")
        if reported_source_count < 4:
            raise RuntimeError("At least four source frames are required.")
        source_count = reported_source_count
        timestamp_mode = "constant-fps-fallback"
        frame_timestamps = np.arange(source_count, dtype=np.float64) / source_fps
        print("WARNING: usable per-frame PTS were unavailable; assuming constant FPS.")

    estimated_duration = max(
        float(container_duration or 0.0),
        float(frame_timestamps[-1]),
    )
    if estimated_duration <= 0:
        estimated_duration = float(
            frame_timestamps[-1] + np.median(np.diff(frame_timestamps))
        )
    if args.max_duration is not None and estimated_duration > args.max_duration + 1e-3:
        raise RuntimeError(
            f"Video duration {estimated_duration:.3f}s exceeds the "
            f"{args.max_duration:.3f}s profile. Split it into overlapping shots."
        )

    cadence = 1.0 / max(args.target_fps, 1e-6)
    desired_times = np.arange(0.0, float(frame_timestamps[-1]) + 1e-9, cadence)
    target_indices = np.unique(
        nearest_frame_indices(frame_timestamps, np.r_[desired_times, frame_timestamps[-1]])
    )
    was_capped = len(target_indices) > args.max_frames
    if was_capped:
        # Cover the complete timeline so late entrants retain a discovery chance.
        target_indices = np.unique(
            nearest_frame_indices(
                frame_timestamps,
                np.linspace(0.0, float(frame_timestamps[-1]), args.max_frames),
            )
        )
    if len(target_indices) < 4:
        target_indices = np.unique(
            np.round(np.linspace(0, source_count - 1, min(source_count, 4))).astype(int)
        )

    frames_dir = args.output / "frames"
    frames_dir.mkdir(parents=True, exist_ok=True)
    for stale_frame in frames_dir.glob("frame_*.jpg"):
        stale_frame.unlink()
    sampled_source: list[int] = []
    timestamps: list[float] = []
    histograms: list[np.ndarray] = []
    letterbox_info: dict | None = None
    target_lookup = {int(source_index): sampled_index for sampled_index, source_index in enumerate(target_indices)}
    decoded_index = 0
    while decoded_index <= int(target_indices[-1]):
        ok, frame = capture.read()
        if not ok:
            break
        if decoded_index not in target_lookup:
            decoded_index += 1
            continue
        sampled_index = target_lookup[decoded_index]
        source_index = decoded_index
        processed, this_info = letterbox(frame, args.size)
        if letterbox_info is None:
            letterbox_info = this_info
        cv2.imwrite(
            str(frames_dir / f"frame_{sampled_index:04d}.jpg"),
            processed,
            [cv2.IMWRITE_JPEG_QUALITY, 95],
        )
        sampled_source.append(int(source_index))
        timestamps.append(float(frame_timestamps[source_index]))
        hsv = cv2.cvtColor(processed, cv2.COLOR_BGR2HSV)
        content_mask = np.zeros(processed.shape[:2], dtype=np.uint8)
        x0, y0, x1, y1 = this_info["contentRectXYXY"]
        content_mask[y0:y1, x0:x1] = 255
        hist = cv2.calcHist(
            [hsv], [0, 1], content_mask, [24, 16], [0, 180, 0, 256]
        )
        histograms.append(cv2.normalize(hist, None).reshape(-1))
        decoded_index += 1
    capture.release()
    if len(sampled_source) != len(target_indices):
        raise RuntimeError(
            f"Decoded only {len(sampled_source)}/{len(target_indices)} selected frames; "
            "the container index and decoder disagree. Transcode to CFR H.264 and retry."
        )

    cut_scores = [0.0]
    for previous, current in zip(histograms, histograms[1:]):
        cut_scores.append(float(cv2.compareHist(previous, current, cv2.HISTCMP_BHATTACHARYYA)))
    suspected_cuts = [index for index, score in enumerate(cut_scores) if score > 0.72]

    discovery = list(range(0, len(sampled_source), max(1, args.discovery_every)))
    if discovery[-1] != len(sampled_source) - 1:
        discovery.append(len(sampled_source) - 1)
    metadata = {
        "sourceVideoName": args.input.name,
        "sourceFps": source_fps,
        "sourceFrameCount": source_count,
        "durationSeconds": float(estimated_duration),
        "timestampMode": timestamp_mode,
        "sampledSourceFrameIndices": sampled_source,
        "sampledTimestampsSeconds": timestamps,
        "effectiveSampleIntervalsSeconds": np.diff(timestamps).round(8).tolist(),
        "discoveryFrameIndices": discovery,
        "letterbox": letterbox_info,
        "shotChangeScores": [round(score, 6) for score in cut_scores],
        "suspectedSceneCutSampleIndices": suspected_cuts,
        "revisions": {"sam3": args.sam3_revision, "spatracker": args.spatracker_revision},
    }
    metadata_path = args.output / "video_metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")
    print(f"Sampled {len(sampled_source)}/{source_count} frames across {metadata['durationSeconds']:.2f}s")
    print(f"Source frame indices: {sampled_source}")
    print(f"SAM 3 discovery sample indices: {discovery}")
    if was_capped:
        print("WARNING: the T4 frame cap forced sparse sampling across the whole clip.")
    if suspected_cuts:
        print(
            "WARNING: possible hard cuts at sampled indices " + str(suspected_cuts)
            + ". A consistent 3D world is only valid within one continuous shot."
        )
    print(metadata_path)


if __name__ == "__main__":
    main()


In [ ]:
shutil.rmtree(FRAME_ROOT, ignore_errors=True)
run([
    sys.executable, WORK_ROOT / "scripts/prepare_video.py",
    "--input", INPUT_VIDEO,
    "--output", FRAME_ROOT,
    "--target-fps", TARGET_FPS,
    "--max-frames", MAX_TRACK_FRAMES,
    "--max-duration", MAX_CLIP_DURATION_SECONDS,
    "--size", PROCESSED_SIZE,
    "--discovery-every", DISCOVERY_EVERY,
    "--sam3-revision", SAM3_REVISION,
    "--spatracker-revision", SPATRACKER_REVISION,
])
VIDEO_METADATA = FRAME_ROOT / "video_metadata.json"
prepared_metadata = json.loads(VIDEO_METADATA.read_text())
if prepared_metadata["durationSeconds"] > MAX_CLIP_DURATION_SECONDS:
    raise RuntimeError(
        f"This T4 profile accepts at most {MAX_CLIP_DURATION_SECONDS:.1f}s per run; "
        "split the video into overlapping continuous clips. Long sparse jumps do not "
        "produce trustworthy SpatialTracker trajectories."
    )
if ABORT_ON_SUSPECTED_SCENE_CUT and prepared_metadata["suspectedSceneCutSampleIndices"]:
    raise RuntimeError(
        "A probable hard cut was found. Trim/split the video into one continuous shot "
        "and rerun so unrelated camera worlds are not fused. Suspect sample indices: "
        + str(prepared_metadata["suspectedSceneCutSampleIndices"])
    )


## 4. کشف دوره‌ای و class-agnostic با SAM 3 Tracker

SAM 3 فقط روی discovery frameها اجرا می‌شود و سپس subprocess آن GPU را آزاد می‌کند. automatic mask generation یک grid نقطه را امتحان می‌کند؛ `SAM3_POINTS_PER_CROP` recall اشیای کوچک در برابر زمان اجرا را کنترل می‌کند و `SAM3_POINTS_PER_BATCH` برای محدودکردن peak VRAM است. mask-NMS و containment suppression duplicateهای واضح و بسیاری از part/wholeها را حذف می‌کنند؛ مرزبندی معنایی «شیء» همچنان یک ابهام ذاتی است.


In [ ]:
%%writefile /content/4d_env/scripts/sam3_discover.py
#!/usr/bin/env python3
"""Class-agnostic SAM 3 mask discovery on sparse video frames.

This uses the official Hugging Face ``Sam3Tracker`` automatic-mask-generation
pipeline. A small point batch controls peak memory; ``points_per_crop`` controls
the speed/recall trade-off without changing the model's native image geometry.
"""

from __future__ import annotations

import argparse
import json
import os
from pathlib import Path

import cv2
import numpy as np
import torch
from huggingface_hub import model_info
from PIL import Image
from transformers import pipeline


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--frames", type=Path, required=True)
    parser.add_argument("--metadata", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--model-id", default="facebook/sam3")
    parser.add_argument("--model-revision", default="main")
    parser.add_argument("--points-per-batch", type=int, default=4)
    parser.add_argument("--points-per-crop", type=int, default=16)
    parser.add_argument("--pred-iou-thresh", type=float, default=0.78)
    parser.add_argument("--stability-thresh", type=float, default=0.86)
    parser.add_argument("--crops-nms-thresh", type=float, default=0.70)
    parser.add_argument("--mask-nms-iou", type=float, default=0.86)
    parser.add_argument("--mask-containment-thresh", type=float, default=0.97)
    parser.add_argument("--min-area", type=float, default=0.001)
    parser.add_argument("--max-area", type=float, default=0.85)
    parser.add_argument("--max-objects", type=int, default=20)
    parser.add_argument("--seed", type=int, default=17)
    return parser.parse_args()


def as_numpy_mask(value: object) -> np.ndarray:
    if isinstance(value, torch.Tensor):
        array = value.detach().cpu().numpy()
    elif isinstance(value, Image.Image):
        array = np.asarray(value)
    else:
        array = np.asarray(value)
    array = np.squeeze(array)
    if array.ndim != 2:
        raise ValueError(f"Expected a 2D mask, got shape {array.shape}")
    return array.astype(bool, copy=False)


def as_score_list(value: object) -> list[float]:
    if isinstance(value, torch.Tensor):
        value = value.detach().float().cpu().numpy()
    return [float(x) for x in np.asarray(value).reshape(-1)]


def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    intersection = np.logical_and(a, b).sum(dtype=np.float64)
    if intersection == 0:
        return 0.0
    union = np.logical_or(a, b).sum(dtype=np.float64)
    return float(intersection / max(union, 1.0))


def smaller_mask_containment(a: np.ndarray, b: np.ndarray) -> float:
    """Return the fraction of the smaller mask covered by the other mask."""
    intersection = np.logical_and(a, b).sum(dtype=np.float64)
    smaller_area = min(a.sum(dtype=np.float64), b.sum(dtype=np.float64))
    return float(intersection / max(smaller_area, 1.0))


def bbox_from_mask(mask: np.ndarray) -> list[float]:
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return [0.0, 0.0, 0.0, 0.0]
    return [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]


def main() -> None:
    args = parse_args()
    if args.points_per_batch < 1 or args.points_per_crop < 2:
        raise ValueError("points-per-batch must be >=1 and points-per-crop >=2")

    meta = json.loads(args.metadata.read_text())
    discovery_indices = [int(x) for x in meta["discoveryFrameIndices"]]
    args.output.mkdir(parents=True, exist_ok=True)
    masks_dir = args.output / "masks"
    masks_dir.mkdir(exist_ok=True)
    for stale_mask in masks_dir.glob("f*-d*.png"):
        stale_mask.unlink()

    processed_size = meta.get("letterbox", {}).get("processedSize")
    if processed_size:
        processed_width, processed_height = map(int, processed_size)
    else:
        processed_width = processed_height = int(meta.get("letterbox", {}).get("size", 518))
    content_rect = meta.get("letterbox", {}).get(
        "contentRectXYXY", [0, 0, processed_width, processed_height]
    )

    if not torch.cuda.is_available():
        raise RuntimeError("SAM 3 discovery requires a CUDA GPU.")
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)

    # T4 is SM75: load the tracker in FP16 and let PyTorch use its available
    # SDPA implementation. No BF16, FlashAttention-3, or custom image resize.
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
    token = os.environ.get("HF_TOKEN")
    resolved_model_revision = model_info(
        args.model_id, revision=args.model_revision, token=token
    ).sha
    if not resolved_model_revision:
        raise RuntimeError("Hugging Face did not resolve an immutable SAM 3 model revision.")
    generator = pipeline(
        "mask-generation",
        model=args.model_id,
        revision=resolved_model_revision,
        token=token,
        device=0,
        dtype=torch.float16,
    )

    output_frames: list[dict] = []
    total_kept = 0
    for frame_index in discovery_indices:
        frame_path = args.frames / f"frame_{frame_index:04d}.jpg"
        image = Image.open(frame_path).convert("RGB")
        height, width = image.height, image.width
        x0, y0, x1, y1 = [int(v) for v in content_rect]
        x0, x1 = np.clip([x0, x1], 0, width)
        y0, y1 = np.clip([y0, y1], 0, height)
        valid_content = np.zeros((height, width), dtype=bool)
        valid_content[y0:y1, x0:x1] = True
        image_area = float(valid_content.sum())
        if image_area <= 0:
            raise RuntimeError("The recorded letterbox content rectangle is empty.")

        with torch.inference_mode():
            outputs = generator(
                image,
                points_per_batch=args.points_per_batch,
                points_per_crop=args.points_per_crop,
                crops_n_layers=0,
                pred_iou_thresh=args.pred_iou_thresh,
                stability_score_thresh=args.stability_thresh,
                crops_nms_thresh=args.crops_nms_thresh,
            )

        raw_masks = [as_numpy_mask(mask) for mask in outputs.get("masks", [])]
        raw_scores = as_score_list(outputs.get("scores", []))
        if len(raw_masks) != len(raw_scores):
            raise RuntimeError(
                f"SAM 3 returned {len(raw_masks)} masks but {len(raw_scores)} scores"
            )

        candidates: list[dict] = []
        for mask, score in zip(raw_masks, raw_scores):
            if not np.isfinite(score):
                continue
            if mask.shape != (height, width):
                mask = cv2.resize(
                    mask.astype(np.uint8), (width, height), interpolation=cv2.INTER_NEAREST
                ).astype(bool)
            raw_area = float(mask.sum(dtype=np.float64))
            mask &= valid_content
            inside_area = float(mask.sum(dtype=np.float64))
            content_coverage = inside_area / max(raw_area, 1.0)
            if content_coverage < 0.80:
                continue
            area_fraction = inside_area / image_area
            if not (args.min_area <= area_fraction <= args.max_area):
                continue
            candidates.append(
                {
                    "score": score,
                    "mask": mask,
                    "areaFraction": area_fraction,
                    "contentCoverage": content_coverage,
                }
            )

        # Prefer the larger region when two pixel masks are effectively nested;
        # this suppresses SAM part/whole duplicates before persistent IDs are
        # created. Distinct visible objects normally occupy disjoint pixels.
        candidates.sort(key=lambda item: item["areaFraction"], reverse=True)
        deduplicated: list[dict] = []
        for candidate in candidates:
            if any(
                mask_iou(candidate["mask"], old["mask"]) >= args.mask_nms_iou
                or smaller_mask_containment(candidate["mask"], old["mask"])
                >= args.mask_containment_thresh
                for old in deduplicated
            ):
                continue
            deduplicated.append(candidate)
        kept = sorted(deduplicated, key=lambda item: item["score"], reverse=True)[
            : args.max_objects
        ]

        frame_record = {
            "frameIndex": frame_index,
            "sourceFrameIndex": int(meta["sampledSourceFrameIndices"][frame_index]),
            "timestampSeconds": float(meta["sampledTimestampsSeconds"][frame_index]),
            "detections": [],
        }
        for local_index, candidate in enumerate(kept):
            det_id = f"f{frame_index:04d}-d{local_index:03d}"
            mask_rel = f"masks/{det_id}.png"
            cv2.imwrite(str(args.output / mask_rel), candidate["mask"].astype(np.uint8) * 255)
            frame_record["detections"].append(
                {
                    "detId": det_id,
                    "frameIndex": frame_index,
                    "sourceFrameIndex": int(meta["sampledSourceFrameIndices"][frame_index]),
                    "timestampSeconds": float(meta["sampledTimestampsSeconds"][frame_index]),
                    "label": "object",
                    "score": round(candidate["score"], 6),
                    "bboxXYXY": bbox_from_mask(candidate["mask"]),
                    "areaFraction": round(candidate["areaFraction"], 8),
                    "contentCoverage": round(candidate["contentCoverage"], 8),
                    "maskPath": mask_rel,
                }
            )
            total_kept += 1
        output_frames.append(frame_record)
        print(f"frame {frame_index:>3}: {len(raw_masks):>3} raw -> {len(kept):>2} masks")

        del outputs, raw_masks
        torch.cuda.empty_cache()

    payload = {
        "schemaVersion": "1.0",
        "kind": "sam3-sparse-automatic-mask-discovery",
        "sam3Revision": meta["revisions"]["sam3"],
        "modelId": args.model_id,
        "requestedModelRevision": args.model_revision,
        "resolvedModelRevision": resolved_model_revision,
        "automaticMaskGeneration": {
            "pointsPerBatch": args.points_per_batch,
            "pointsPerCrop": args.points_per_crop,
            "predIouThreshold": args.pred_iou_thresh,
            "stabilityThreshold": args.stability_thresh,
            "cropsNmsThreshold": args.crops_nms_thresh,
            "maskNmsIou": args.mask_nms_iou,
            "maskContainmentThreshold": args.mask_containment_thresh,
            "minimumAreaFraction": args.min_area,
            "maximumAreaFraction": args.max_area,
            "maximumObjectsPerFrame": args.max_objects,
            "randomSeed": args.seed,
        },
        "frames": output_frames,
    }
    (args.output / "detections.json").write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + "\n"
    )
    print(f"Saved {total_kept} discovery masks to {args.output}")


if __name__ == "__main__":
    main()


In [ ]:
sam_environment = os.environ.copy()
sam_environment["HF_TOKEN"] = hf_token
sam_environment["PYTHONHASHSEED"] = str(RANDOM_SEED)
sam_environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
shutil.rmtree(SAM_ROOT, ignore_errors=True)
run([
    SAM_PY, WORK_ROOT / "scripts/sam3_discover.py",
    "--frames", FRAME_ROOT / "frames",
    "--metadata", VIDEO_METADATA,
    "--output", SAM_ROOT,
    "--model-id", SAM3_MODEL_ID,
    "--model-revision", SAM3_MODEL_REVISION,
    "--points-per-batch", SAM3_POINTS_PER_BATCH,
    "--points-per-crop", SAM3_POINTS_PER_CROP,
    "--pred-iou-thresh", SAM3_PRED_IOU_THRESHOLD,
    "--stability-thresh", SAM3_STABILITY_THRESHOLD,
    "--min-area", SAM3_MIN_AREA_FRACTION,
    "--max-objects", MAX_OBJECTS_PER_DISCOVERY,
    "--seed", RANDOM_SEED,
], env=sam_environment)
SAM_DETECTIONS = SAM_ROOT / "detections.json"


## 5. ساخت query با birth time واقعی

از mask فرسایش‌یافته نقاط پراکنده انتخاب می‌شوند. یک object که اولین‌بار وسط فیلم دیده شده queryهایی با همان index زمانی می‌گیرد؛ بنابراین قبل از تولد، خروجی آن معتبر شمرده نمی‌شود. reappearance نیز seedهای تازه تولید می‌کند که بعداً می‌توانند به همان world ID وصل شوند.


In [ ]:
%%writefile /content/4d_env/scripts/build_queries.py
#!/usr/bin/env python3
"""Turn sparse SAM 3 masks into timestamped SpaTrackerV2 point queries."""

from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path

import cv2
import numpy as np


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--detections", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--points-per-object", type=int, default=6)
    parser.add_argument("--max-total", type=int, default=384)
    parser.add_argument("--erosion", type=int, default=5)
    return parser.parse_args()


def stable_seed(text: str) -> int:
    return int(hashlib.sha1(text.encode("utf-8")).hexdigest()[:8], 16)


def diverse_points(mask: np.ndarray, count: int, erosion: int, seed: int) -> np.ndarray:
    kernel_size = max(1, erosion)
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    interior = cv2.erode(mask.astype(np.uint8), kernel, iterations=1) > 0
    if interior.sum() < count:
        interior = mask.astype(bool)
    ys, xs = np.where(interior)
    if len(xs) == 0:
        return np.empty((0, 2), dtype=np.float32)

    points = np.column_stack([xs, ys]).astype(np.float32)
    rng = np.random.default_rng(seed)
    if len(points) > 4000:
        points = points[rng.choice(len(points), size=4000, replace=False)]

    centroid = points.mean(axis=0)
    selected = [int(np.argmin(((points - centroid) ** 2).sum(axis=1)))]
    min_distance = ((points - points[selected[0]]) ** 2).sum(axis=1)
    while len(selected) < min(count, len(points)):
        next_index = int(np.argmax(min_distance))
        selected.append(next_index)
        distance = ((points - points[next_index]) ** 2).sum(axis=1)
        min_distance = np.minimum(min_distance, distance)
    result = points[selected]
    if len(result) < count:
        extra = result[rng.choice(len(result), size=count - len(result), replace=True)]
        result = np.concatenate([result, extra], axis=0)
    return result[:count]


def main() -> None:
    args = parse_args()
    payload = json.loads(args.detections.read_text())
    root = args.detections.parent
    frame_detections = [
        sorted(frame["detections"], key=lambda d: -d["score"])
        for frame in payload["frames"]
    ]
    capacity = max(1, args.max_total // max(args.points_per_object, 1))
    # Fair round-robin allocation prevents early frames from consuming the whole
    # T4 query budget and hiding objects that enter near the end of the clip.
    detections: list[dict] = []
    for rank in range(max((len(items) for items in frame_detections), default=0)):
        for items in frame_detections:
            if rank < len(items):
                detections.append(items[rank])
                if len(detections) >= capacity:
                    break
        if len(detections) >= capacity:
            break
    detections.sort(key=lambda d: (d["frameIndex"], -d["score"]))

    queries: list[list[float]] = []
    det_ids: list[str] = []
    local_indices: list[int] = []
    for detection in detections:
        mask = cv2.imread(str(root / detection["maskPath"]), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(root / detection["maskPath"])
        points = diverse_points(
            mask > 0,
            count=args.points_per_object,
            erosion=args.erosion,
            seed=stable_seed(detection["detId"]),
        )
        for local_index, (x, y) in enumerate(points):
            queries.append([float(detection["frameIndex"]), float(x), float(y)])
            det_ids.append(detection["detId"])
            local_indices.append(local_index)

    if not queries:
        raise RuntimeError(
            "SAM 3 returned no usable objects. Lower the mask-quality thresholds, "
            "increase SAM3_POINTS_PER_CROP, or lower the minimum mask area."
        )
    total_detections = sum(len(items) for items in frame_detections)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        args.output,
        queries=np.asarray(queries, dtype=np.float32),
        det_ids=np.asarray(det_ids, dtype="U32"),
        local_indices=np.asarray(local_indices, dtype=np.int32),
        points_per_object=np.asarray(args.points_per_object, dtype=np.int32),
        maximum_queries=np.asarray(args.max_total, dtype=np.int32),
        selected_detection_count=np.asarray(len(detections), dtype=np.int32),
        available_detection_count=np.asarray(total_detections, dtype=np.int32),
    )
    if len(detections) < total_detections:
        print(
            f"Query cap kept {len(detections)}/{total_detections} masks with fair "
            "coverage across discovery frames."
        )
    print(f"Saved {len(queries)} timestamped point queries to {args.output}")


if __name__ == "__main__":
    main()


In [ ]:
run([
    sys.executable, WORK_ROOT / "scripts/build_queries.py",
    "--detections", SAM_DETECTIONS,
    "--output", QUERY_PATH,
    "--points-per-object", POINTS_PER_OBJECT,
    "--max-total", MAX_TOTAL_QUERIES,
])


## 6. SpatialTrackerV2 Online در FP16

front model ابتدا depth/camera/confidence را می‌سازد؛ seedهای depth-valid با mapping صریح نگه داشته می‌شوند. سپس front model آزاد و tracker online بارگذاری می‌شود. `query_no_BA=True` مانع خراب‌شدن bundle adjustment دوربین توسط نقاط object متحرک است.


In [ ]:
%%writefile /content/4d_env/scripts/run_spatracker.py
#!/usr/bin/env python3
"""T4-safe, ID-preserving SpaTrackerV2 run for timestamped object seeds."""

from __future__ import annotations

import argparse
import gc
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--repo", type=Path, required=True)
    parser.add_argument("--frames", type=Path, required=True)
    parser.add_argument("--metadata", type=Path, required=True)
    parser.add_argument("--queries", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--support-points", type=int, default=128)
    parser.add_argument("--iterations", type=int, default=2)
    parser.add_argument("--min-depth-confidence", type=float, default=0.5)
    parser.add_argument("--seed", type=int, default=17)
    parser.add_argument(
        "--front-revision", default="2e7164a93770cb934861d7de579bb81f58ad0b03"
    )
    parser.add_argument(
        "--tracker-revision", default="d1e3b89e0ab10b604b0caa1cd001d8c24182bd86"
    )
    return parser.parse_args()


def load_frames(frame_dir: Path, count: int) -> torch.Tensor:
    frames = []
    for frame_index in range(count):
        path = frame_dir / f"frame_{frame_index:04d}.jpg"
        frames.append(np.asarray(Image.open(path).convert("RGB"), dtype=np.uint8).copy())
    return torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2).float()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def prefilter_queries(
    queries: np.ndarray,
    det_ids: np.ndarray,
    tracker_confidence: torch.Tensor,
    source_size: tuple[int, int],
    sample_features5d,
    min_confidence: float,
    minimum_per_detection: int = 3,
) -> np.ndarray:
    source_height, source_width = source_size
    tracker_height, tracker_width = tracker_confidence.shape[-2:]
    scaled_queries = queries.copy()
    scaled_queries[:, 1] *= tracker_width / source_width
    scaled_queries[:, 2] *= tracker_height / source_height
    sampled_confidence = (
        sample_features5d(
            tracker_confidence[None],
            torch.from_numpy(scaled_queries).float()[None, None],
        )
        .reshape(-1)
        .cpu()
        .numpy()
    )
    threshold = max(float(min_confidence), 0.5)
    keep: list[int] = []
    for det_id in np.unique(det_ids):
        indices = np.flatnonzero(det_ids == det_id)
        preferred = [
            int(index)
            for index in indices
            if np.isfinite(sampled_confidence[index])
            and sampled_confidence[index] > threshold
        ]
        if len(preferred) < minimum_per_detection:
            print(
                f"Dropping {det_id}: only {len(preferred)} depth-confident seeds "
                f"(<{minimum_per_detection})."
            )
            continue
        keep.extend(preferred)
    return np.asarray(sorted(set(keep)), dtype=np.int64)


def main() -> None:
    args = parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError("SpaTrackerV2 requires a CUDA GPU.")
    sys.path.insert(0, str(args.repo))
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    torch.backends.cudnn.benchmark = False

    from models.SpaTrackV2.models.predictor import Predictor
    from models.SpaTrackV2.models.vggt4track.models.vggt_moe import VGGT4Track
    from models.SpaTrackV2.models.vggt4track.utils.load_fn import preprocess_image
    from models.SpaTrackV2.utils.model_utils import sample_features5d
    from huggingface_hub import hf_hub_download
    import utils3d

    metadata = json.loads(args.metadata.read_text())
    frame_count = len(metadata["sampledSourceFrameIndices"])
    video = load_frames(args.frames, frame_count)
    if tuple(video.shape[-2:]) != (518, 518):
        raise ValueError(
            f"This pinned SpatialTrackerV2 Front path requires 518x518 frames, got {tuple(video.shape[-2:])}."
        )
    query_payload = np.load(args.queries)
    queries_all = query_payload["queries"].astype(np.float32)
    det_ids_all = query_payload["det_ids"].astype(str)

    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)

    print("Loading SpatialTrackerV2 front model...")
    front = VGGT4Track.from_pretrained(
        "Yuxihenry/SpatialTrackerV2_Front", revision=args.front_revision
    )
    front.eval()
    front.to("cuda")
    front_video = preprocess_image(video, mode="pad", target_size=518)[None]
    with torch.inference_mode(), torch.autocast(
        device_type="cuda", dtype=torch.float16
    ):
        predictions = front(front_video.cuda() / 255.0)

    c2w_front = predictions["poses_pred"].squeeze(0).float().cpu().numpy()
    intrinsics_front = predictions["intrs"].squeeze(0).float().cpu().numpy()
    point_map_front = predictions["points_map"].squeeze(0).float().cpu().numpy()
    depth_front = point_map_front[..., 2]
    depth_conf_front = predictions["unc_metric"].squeeze(0).float().cpu().numpy()
    content_rect = metadata.get("letterbox", {}).get(
        "contentRectXYXY", [0, 0, depth_front.shape[2], depth_front.shape[1]]
    )
    x0, y0, x1, y1 = [int(v) for v in content_rect]
    content_mask = np.zeros(depth_front.shape[1:], dtype=np.float32)
    content_mask[
        max(0, y0) : min(depth_front.shape[1], y1),
        max(0, x0) : min(depth_front.shape[2], x1),
    ] = 1.0
    depth_conf_front *= content_mask[None]
    depth_front = np.where(content_mask[None] > 0, depth_front, 0.0)

    tracker_config_path = hf_hub_download(
        repo_id="Yuxihenry/SpatialTrackerV2-Online",
        revision=args.tracker_revision,
        filename="config.json",
    )
    tracker_config = json.loads(Path(tracker_config_path).read_text())
    tracker_resolution = int(tracker_config["args"]["resolution"])
    if tracker_resolution <= 0 or tracker_resolution % 14:
        raise RuntimeError(f"Unexpected tracker resolution: {tracker_resolution}")

    # Reproduce the stage-1 seed-validity mask at the tracker's internal 336px
    # resolution, including its nearest depth/confidence resize and bilinear
    # sampling at rescaled query coordinates. The upstream filter is patched out
    # only because it loses query IDs; validation itself remains equivalent.
    depth_tensor = torch.nn.functional.interpolate(
        torch.from_numpy(depth_front).float()[:, None],
        size=(tracker_resolution, tracker_resolution),
        mode="nearest",
    )
    confidence_tensor = torch.nn.functional.interpolate(
        torch.from_numpy(depth_conf_front).float()[:, None],
        size=(tracker_resolution, tracker_resolution),
        mode="nearest",
    )
    positive_values = depth_tensor[depth_tensor > 0].reshape(-1)
    if len(positive_values) == 0:
        raise RuntimeError("The front model returned no positive depth.")
    q25 = torch.kthvalue(positive_values, max(1, int(0.25 * len(positive_values)))).values
    q75 = torch.kthvalue(positive_values, max(1, int(0.75 * len(positive_values)))).values
    upper_bound = (q75 + 0.8 * (q75 - q25)).clamp(min=1e-6, max=10 * q25)
    depth_roi = (depth_tensor > 1e-1) & (depth_tensor < upper_bound)
    depth_edges = utils3d.torch.depth_edge(
        depth_tensor, rtol=0.03, mask=depth_roi
    )
    tracker_seed_confidence = confidence_tensor * (
        depth_roi & (~depth_edges)
    )

    kept_query_indices = prefilter_queries(
        queries_all,
        det_ids_all,
        tracker_seed_confidence,
        tuple(video.shape[-2:]),
        sample_features5d,
        args.min_depth_confidence,
    )
    if len(kept_query_indices) == 0:
        raise RuntimeError("No SAM seed has usable depth. Try a different clip or lower the threshold.")
    queries = queries_all[kept_query_indices]
    print(f"Depth-valid object seeds: {len(queries)}/{len(queries_all)}")

    # Nothing from the front model remains on GPU when the tracker is loaded.
    del predictions, front, front_video
    gc.collect()
    torch.cuda.empty_cache()

    print("Loading SpatialTrackerV2 online tracker...")
    tracker = Predictor.from_pretrained(
        "Yuxihenry/SpatialTrackerV2-Online", revision=args.tracker_revision
    )
    if int(tracker.spatrack.resolution) != tracker_resolution:
        raise RuntimeError(
            f"Tracker checkpoint resolution {tracker.spatrack.resolution} does not match "
            f"the pinned repo config {tracker_resolution}."
        )
    tracker.spatrack.track_num = int(args.support_points)
    tracker.eval()
    tracker.to("cuda")

    try:
        with torch.inference_mode(), torch.autocast(
            device_type="cuda", dtype=torch.float16
        ):
            (
                c2w,
                intrinsics,
                point_map,
                depth_confidence,
                track3d_camera,
                track2d,
                visibility,
                track_confidence,
                _video_out,
            ) = tracker.forward(
                video,
                depth=depth_front,
                intrs=intrinsics_front,
                extrs=c2w_front,
                unc_metric=depth_conf_front,
                queries=queries,
                fps=1,
                full_point=False,
                iters_track=int(args.iterations),
                query_no_BA=True,
                fixed_cam=False,
                stage=1,
                support_frame=len(video) - 1,
                replace_ratio=0.2,
            )
    except torch.OutOfMemoryError as exc:
        raise RuntimeError(
            "T4 ran out of memory. Lower MAX_TRACK_FRAMES, MAX_TOTAL_QUERIES, "
            "SUPPORT_POINTS, or TRACK_ITERATIONS and rerun from preprocessing."
        ) from exc

    c2w_cpu = c2w.float().cpu()
    xyz_camera = track3d_camera[:, :, :3].float().cpu()
    xyz_world = (
        torch.einsum("tij,tnj->tni", c2w_cpu[:, :3, :3], xyz_camera)
        + c2w_cpu[:, None, :3, 3]
    )

    args.output.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        args.output,
        coords_world=xyz_world.numpy().astype(np.float32),
        coords_camera=xyz_camera.numpy().astype(np.float32),
        track2d=track2d.float().cpu().numpy().astype(np.float32),
        visibility=visibility.float().cpu().numpy().astype(np.float32),
        track_confidence=track_confidence.float().cpu().numpy().astype(np.float32),
        c2w=c2w_cpu.numpy().astype(np.float32),
        w2c=np.linalg.inv(c2w_cpu.numpy()).astype(np.float32),
        intrinsics=intrinsics.float().cpu().numpy().astype(np.float32),
        point_map_camera=point_map.float().cpu().numpy().astype(np.float32),
        depth_confidence=depth_confidence.float().cpu().numpy().astype(np.float32),
        kept_query_indices=kept_query_indices,
        front_model_revision=np.asarray(args.front_revision),
        tracker_model_revision=np.asarray(args.tracker_revision),
        random_seed=np.asarray(args.seed, dtype=np.int64),
        patched_attention_sha256=np.asarray(
            sha256_file(
                args.repo / "models/SpaTrackV2/models/tracker3D/co_tracker/utils.py"
            )
        ),
        patched_query_path_sha256=np.asarray(
            sha256_file(args.repo / "models/SpaTrackV2/models/SpaTrack.py")
        ),
        sampled_source_frame_indices=np.asarray(
            metadata["sampledSourceFrameIndices"], dtype=np.int64
        ),
        timestamps_seconds=np.asarray(
            metadata["sampledTimestampsSeconds"], dtype=np.float64
        ),
    )
    print(f"Saved SpatialTrackerV2 outputs to {args.output}")


if __name__ == "__main__":
    main()


In [ ]:
spatial_environment = os.environ.copy()
spatial_environment["HF_TOKEN"] = hf_token
spatial_environment["PYTHONHASHSEED"] = str(RANDOM_SEED)
spatial_environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
run([
    SPA_PY, WORK_ROOT / "scripts/run_spatracker.py",
    "--repo", SPA_REPO,
    "--frames", FRAME_ROOT / "frames",
    "--metadata", VIDEO_METADATA,
    "--queries", QUERY_PATH,
    "--output", TRACK_PATH,
    "--support-points", SUPPORT_POINTS,
    "--iterations", TRACK_ITERATIONS,
    "--front-revision", SPATRACKER_FRONT_REVISION,
    "--tracker-revision", SPATRACKER_ONLINE_REVISION,
    "--seed", RANDOM_SEED,
], env=spatial_environment)


## 7. Re-ID، OBB، prediction و export

detectionهای مستقل SAM با Hungarian matching و ترکیب point-inside-mask، 2D box، appearance، 3D و label به identity پایدار وصل می‌شوند. هندسه‌ی mask+depth برای اندازه‌ی OBB و point trackها برای حرکت استفاده می‌شود. در نبود measurement، Kalman constant-velocity با uncertainty افزاینده pose را نگه می‌دارد؛ پس entity هنگام occlusion ناپدید نمی‌شود.

فایل strict از unit-cube محلی استفاده می‌کند: `position=center`، `rotation=quaternion xyzw` و `scale=dimensions`. برای آبجکت دیررس، `scale=[0,0,0]` تا birth و بعد اندازه‌ی واقعی است؛ هنگام occlusion scale صفر نمی‌شود.

کران برخورد از تمام نقطه‌های معتبر mask/depth و track ساخته می‌شود؛ برخلاف OBB نمایشی، percentile، erosion و subsampling نقاط را از این کران حذف نمی‌کنند. پس از فیلتر pose نیز کران در مختصات pose خروجی بازساخته می‌شود. margin برابر بیشینهٔ `1e-4` واحد خروجی و یک درصد قطر OBB نمایشی است. تضمین فقط شامل هندسهٔ مشاهده‌شده در زمان‌های نمونه است، نه سطح پنهان یا خطای prediction.


In [ ]:
%%writefile /content/4d_env/scripts/fuse_and_export.py
#!/usr/bin/env python3
"""Fuse SAM discoveries and SpaTracker points into strict env-v1 + rich sidecar."""

from __future__ import annotations

import argparse
import hashlib
import json
import math
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.spatial.transform import Rotation

matplotlib.use("Agg")
import matplotlib.pyplot as plt


GENERIC_LABELS = {"object", "objects", "thing", "things", "item", "items"}
PALETTE = [
    "#4aa8ff", "#ff9e3d", "#52c77b", "#d66bea", "#f25f5c", "#46c5c1",
    "#e0c34f", "#8d99ff", "#f58fb0", "#78b84a", "#ba8cff", "#e07a5f",
]


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--metadata", type=Path, required=True)
    parser.add_argument("--detections", type=Path, required=True)
    parser.add_argument("--queries", type=Path, required=True)
    parser.add_argument("--tracks", type=Path, required=True)
    parser.add_argument("--frames", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--meters-per-unit", type=float, default=None)
    parser.add_argument("--visibility", type=float, default=0.95)
    parser.add_argument("--track-confidence", type=float, default=0.35)
    parser.add_argument("--association", type=float, default=0.50)
    parser.add_argument("--max-reid-gap-seconds", type=float, default=8.0)
    parser.add_argument("--max-prediction-seconds", type=float, default=4.0)
    parser.add_argument("--environment-id", default="video-environment-001")
    return parser.parse_args()


def normalize(v: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    return v / max(float(np.linalg.norm(v)), eps)


def rotation_between(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a, b = normalize(a), normalize(b)
    cross = np.cross(a, b)
    sine = float(np.linalg.norm(cross))
    cosine = float(np.clip(np.dot(a, b), -1.0, 1.0))
    if sine < 1e-8:
        if cosine > 0:
            return np.eye(3)
        axis = normalize(np.cross(a, np.array([1.0, 0.0, 0.0])))
        if np.linalg.norm(axis) < 1e-8:
            axis = normalize(np.cross(a, np.array([0.0, 0.0, 1.0])))
        return Rotation.from_rotvec(axis * math.pi).as_matrix()
    k = np.array(
        [[0.0, -cross[2], cross[1]], [cross[2], 0.0, -cross[0]], [-cross[1], cross[0], 0.0]]
    )
    return np.eye(3) + k + k @ k * ((1.0 - cosine) / (sine * sine))


def transform_points(points: np.ndarray, rotation: np.ndarray, translation: np.ndarray) -> np.ndarray:
    points = np.asarray(points)
    return points @ rotation.T + translation


def load_mask(detection: dict, detection_root: Path) -> np.ndarray:
    mask = cv2.imread(str(detection_root / detection["maskPath"]), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(detection_root / detection["maskPath"])
    return mask > 0


def masked_histogram(image_bgr: np.ndarray, mask: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], mask.astype(np.uint8), [16, 8], [0, 180, 0, 256])
    hist = hist.reshape(-1).astype(np.float64)
    norm = np.linalg.norm(hist)
    return hist / norm if norm > 0 else hist


def bbox_iou(a: list[float] | np.ndarray, b: list[float] | np.ndarray) -> float:
    ax0, ay0, ax1, ay1 = [float(x) for x in a]
    bx0, by0, bx1, by1 = [float(x) for x in b]
    ix0, iy0, ix1, iy1 = max(ax0, bx0), max(ay0, by0), min(ax1, bx1), min(ay1, by1)
    intersection = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    union = max(0.0, ax1 - ax0) * max(0.0, ay1 - ay0) + max(0.0, bx1 - bx0) * max(0.0, by1 - by0) - intersection
    return intersection / union if union > 0 else 0.0


def thresholded_assignment(scores: np.ndarray, threshold: float) -> list[tuple[int, int]]:
    """Maximum-score one-to-one matches, with an explicit unmatched option."""
    scores = np.asarray(scores, dtype=np.float64)
    if scores.ndim != 2 or scores.size == 0:
        return []
    row_count, column_count = scores.shape
    assignment_cost = np.zeros(
        (row_count + column_count, row_count + column_count), dtype=np.float64
    )
    assignment_cost[:row_count, :column_count] = 1e6
    eligible = np.isfinite(scores) & (scores >= float(threshold))
    real_cost = assignment_cost[:row_count, :column_count]
    real_cost[eligible] = -scores[eligible]
    rows, columns = linear_sum_assignment(assignment_cost)
    return [
        (int(row), int(column))
        for row, column in zip(rows, columns)
        if row < row_count and column < column_count and eligible[row, column]
    ]


def points_bbox(points: np.ndarray) -> list[float]:
    if len(points) == 0:
        return [0.0, 0.0, 0.0, 0.0]
    return [
        float(points[:, 0].min()), float(points[:, 1].min()),
        float(points[:, 0].max() + 1), float(points[:, 1].max() + 1),
    ]


def robust_points(points: np.ndarray, z_limit: float = 4.5) -> np.ndarray:
    points = np.asarray(points, dtype=np.float64)
    points = points[np.isfinite(points).all(axis=1)]
    if len(points) < 4:
        return points
    median = np.median(points, axis=0)
    distance = np.linalg.norm(points - median, axis=1)
    mad = np.median(np.abs(distance - np.median(distance))) + 1e-8
    keep = distance <= np.median(distance) + z_limit * 1.4826 * mad
    return points[keep]


def fit_yaw_box(points: np.ndarray, previous_yaw: float | None = None) -> dict | None:
    points = robust_points(points)
    if len(points) == 0:
        return None
    median = np.median(points, axis=0)
    xz = points[:, [0, 2]] - median[[0, 2]]
    covariance = xz.T @ xz / max(len(xz) - 1, 1)
    values, vectors = np.linalg.eigh(covariance)
    major = vectors[:, int(np.argmax(values))]
    yaw = math.atan2(float(major[0]), float(major[1]))
    if previous_yaw is not None:
        candidates = [yaw + k * math.pi for k in range(-3, 4)]
        yaw = min(candidates, key=lambda value: abs(value - previous_yaw))
    rot = Rotation.from_euler("y", yaw).as_matrix()
    local = (points - median) @ rot
    low, high = np.quantile(local, [0.05, 0.95], axis=0)
    local_center = (low + high) / 2.0
    center = median + rot @ local_center
    size = np.maximum(high - low, max(float(np.max(high - low)) * 0.04, 1e-3))
    return {"center": center, "size": size, "yaw": yaw, "pointCount": int(len(points))}


def ransac_ground(points: np.ndarray, seed: int = 7) -> tuple[np.ndarray, float, dict]:
    points = robust_points(points, z_limit=8.0)
    if len(points) < 100:
        return np.eye(3), 0.0, {"status": "fallback-first-camera-up", "inlierRatio": 0.0}
    rng = np.random.default_rng(seed)
    scale = max(float(np.median(np.linalg.norm(points, axis=1))), 1e-3)
    threshold = max(0.012 * scale, 1e-3)
    best_indices = np.empty(0, dtype=np.int64)
    best_normal = np.array([0.0, 1.0, 0.0])
    for _ in range(350):
        sample = points[rng.choice(len(points), 3, replace=False)]
        normal = np.cross(sample[1] - sample[0], sample[2] - sample[0])
        if np.linalg.norm(normal) < 1e-8:
            continue
        normal = normalize(normal)
        if abs(float(normal[1])) < 0.55:
            continue
        if normal[1] < 0:
            normal = -normal
        offset = -float(normal @ sample[0])
        distance = np.abs(points @ normal + offset)
        indices = np.flatnonzero(distance < threshold)
        if len(indices) > len(best_indices):
            best_indices, best_normal = indices, normal
    ratio = len(best_indices) / max(len(points), 1)
    if len(best_indices) < 80 or ratio < 0.04:
        return np.eye(3), 0.0, {"status": "fallback-first-camera-up", "inlierRatio": round(ratio, 5)}
    alignment = rotation_between(best_normal, np.array([0.0, 1.0, 0.0]))
    aligned = points[best_indices] @ alignment.T
    ground_y = float(np.median(aligned[:, 1]))
    return alignment, ground_y, {
        "status": "estimated-ground-plane",
        "inlierRatio": round(ratio, 5),
        "normalBeforeAlignment": best_normal.round(7).tolist(),
    }


def metric_scale_metadata(meters_per_unit: float | None) -> dict:
    """Record measured calibration; absence never implies one meter per unit."""
    if meters_per_unit is None:
        return {"status": "relative"}
    if not math.isfinite(meters_per_unit) or meters_per_unit <= 0:
        raise ValueError("--meters-per-unit must be finite and positive")
    # The measured factor has already been applied: one exported unit is one meter.
    return {"status": "calibrated", "metersPerUnit": 1.0}


def collision_bounds_in_pose(
    world_bounds: tuple[np.ndarray, np.ndarray], center: np.ndarray,
    rotation: np.ndarray, size: np.ndarray, margin: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Enclose an observed world AABB in the exact exported unit-cube transform."""
    low, high = world_bounds
    corners = np.asarray([[x, y, z] for x in (low[0], high[0])
                          for y in (low[1], high[1]) for z in (low[2], high[2])])
    local = ((corners - center) @ rotation) / size
    padding = margin / size
    return local.min(axis=0) - padding, local.max(axis=0) + padding


class ConstantVelocityFilter:
    def __init__(self, process_variance: float = 0.08):
        self.x: np.ndarray | None = None
        self.p = np.eye(6)
        self.q = process_variance

    def predict(self, dt: float) -> None:
        if self.x is None:
            return
        f = np.eye(6)
        f[:3, 3:] = np.eye(3) * dt
        g = np.vstack([np.eye(3) * (0.5 * dt * dt), np.eye(3) * dt])
        self.x = f @ self.x
        self.p = f @ self.p @ f.T + g @ (np.eye(3) * self.q) @ g.T
        self.p = 0.5 * (self.p + self.p.T)

    def update(self, measurement: np.ndarray, variance: float) -> None:
        measurement = np.asarray(measurement, dtype=np.float64)
        if self.x is None:
            self.x = np.r_[measurement, np.zeros(3)]
            self.p = np.eye(6) * max(variance, 0.02)
            return
        h = np.zeros((3, 6))
        h[:, :3] = np.eye(3)
        r = np.eye(3) * max(variance, 1e-4)
        innovation = measurement - h @ self.x
        s = h @ self.p @ h.T + r
        # Solve S K^T = H P rather than explicitly forming S^{-1}.
        gain = np.linalg.solve(s, h @ self.p).T
        self.x = self.x + gain @ innovation
        correction = np.eye(6) - gain @ h
        # Joseph form preserves positive semidefiniteness under roundoff.
        self.p = correction @ self.p @ correction.T + gain @ r @ gain.T
        self.p = 0.5 * (self.p + self.p.T)


def compress_timeline(samples: list[dict]) -> list[dict]:
    if not samples:
        return []
    ranges: list[dict] = []
    start = samples[0]
    last = start
    for sample in samples[1:]:
        if sample["state"] != start["state"] or sample["poseSource"] != start["poseSource"]:
            ranges.append(
                {
                    "startFrame": start["sourceFrame"], "endFrame": last["sourceFrame"],
                    "startT": start["t"], "endT": last["t"],
                    "state": start["state"], "poseSource": start["poseSource"],
                }
            )
            start = sample
        last = sample
    ranges.append(
        {
            "startFrame": start["sourceFrame"], "endFrame": last["sourceFrame"],
            "startT": start["t"], "endT": last["t"],
            "state": start["state"], "poseSource": start["poseSource"],
        }
    )
    return ranges


def rounded(values: np.ndarray | list[float], digits: int = 6) -> list[float]:
    return [round(float(x), digits) for x in np.asarray(values).reshape(-1)]


def main() -> None:
    args = parse_args()
    if args.meters_per_unit is not None and (
        not math.isfinite(args.meters_per_unit) or args.meters_per_unit <= 0
    ):
        raise ValueError("--meters-per-unit must be finite and positive")
    args.output.mkdir(parents=True, exist_ok=True)
    metadata = json.loads(args.metadata.read_text())
    discovery_payload = json.loads(args.detections.read_text())
    query_payload = np.load(args.queries)
    result = np.load(args.tracks)

    detections = [d for frame in discovery_payload["frames"] for d in frame["detections"]]
    detection_by_id = {d["detId"]: d for d in detections}
    detections_by_frame: dict[int, list[dict]] = defaultdict(list)
    for detection in detections:
        detections_by_frame[int(detection["frameIndex"])].append(detection)

    kept = result["kept_query_indices"].astype(np.int64)
    queries = query_payload["queries"][kept].astype(np.float64)
    query_det_ids = query_payload["det_ids"][kept].astype(str)
    det_to_queries: dict[str, np.ndarray] = {
        det_id: np.flatnonzero(query_det_ids == det_id) for det_id in np.unique(query_det_ids)
    }
    # Ignore masks that ended up with no usable 3D seed.
    detections = [d for d in detections if d["detId"] in det_to_queries]
    detections_by_frame = defaultdict(list)
    for detection in detections:
        detections_by_frame[int(detection["frameIndex"])].append(detection)

    xyz_world = result["coords_world"].astype(np.float64)
    track2d = result["track2d"].astype(np.float64)
    visibility = result["visibility"].reshape(xyz_world.shape[:2]).astype(np.float64)
    confidence = result["track_confidence"].reshape(xyz_world.shape[:2]).astype(np.float64)
    c2w = result["c2w"].astype(np.float64)
    w2c = result["w2c"].astype(np.float64)
    intrinsics = result["intrinsics"].astype(np.float64)
    point_map = result["point_map_camera"].astype(np.float64)
    depth_confidence = result["depth_confidence"].astype(np.float64)
    timestamps = result["timestamps_seconds"].astype(np.float64)
    source_frames = result["sampled_source_frame_indices"].astype(np.int64)
    frame_count, height, width = point_map.shape[0], point_map.shape[2], point_map.shape[3]
    content_rect = metadata.get("letterbox", {}).get(
        "contentRectXYXY", [0, 0, width, height]
    )
    content_x0, content_y0, content_x1, content_y1 = [int(v) for v in content_rect]
    content_x0, content_x1 = np.clip([content_x0, content_x1], 0, width)
    content_y0, content_y1 = np.clip([content_y0, content_y1], 0, height)
    valid_content_mask = np.zeros((height, width), dtype=bool)
    valid_content_mask[content_y0:content_y1, content_x0:content_x1] = True
    if not valid_content_mask.any():
        raise RuntimeError("The recorded letterbox content rectangle is empty.")

    detection_root = args.detections.parent
    mask_cache = {d["detId"]: load_mask(d, detection_root) for d in detections}
    image_cache: dict[int, np.ndarray] = {}
    hist_cache: dict[str, np.ndarray] = {}
    for detection in detections:
        frame_index = int(detection["frameIndex"])
        if frame_index not in image_cache:
            image_cache[frame_index] = cv2.imread(
                str(args.frames / f"frame_{frame_index:04d}.jpg"), cv2.IMREAD_COLOR
            )
        hist_cache[detection["detId"]] = masked_histogram(
            image_cache[frame_index], mask_cache[detection["detId"]]
        )

    # Put tracker world into the first-camera frame, then convert OpenCV
    # (+x right,+y down,+z forward) to env (+x right,+y up,-z forward).
    basis = np.diag([1.0, -1.0, -1.0])
    first_w2c = w2c[0]
    base_rotation = basis @ first_w2c[:3, :3]
    base_translation = basis @ first_w2c[:3, 3]

    # Estimate gravity from lower-image point clouds. Object masks on discovery
    # frames are excluded to favor floor/support-plane points.
    ground_candidates: list[np.ndarray] = []
    rng = np.random.default_rng(11)
    for t in range(frame_count):
        valid = (depth_confidence[t] > 0.5) & valid_content_mask
        lower_content_y = content_y0 + int((content_y1 - content_y0) * 0.55)
        valid[:lower_content_y] = False
        for detection in detections_by_frame.get(t, []):
            valid &= ~mask_cache[detection["detId"]]
        ys, xs = np.where(valid)
        if len(xs) > 1600:
            chosen = rng.choice(len(xs), 1600, replace=False)
            ys, xs = ys[chosen], xs[chosen]
        if len(xs) == 0:
            continue
        camera_points = point_map[t].transpose(1, 2, 0)[ys, xs]
        world_points = transform_points(camera_points, c2w[t, :3, :3], c2w[t, :3, 3])
        preliminary = transform_points(world_points, base_rotation, base_translation)
        ground_candidates.append(preliminary)
    ground_points = np.concatenate(ground_candidates, axis=0) if ground_candidates else np.empty((0, 3))
    gravity_rotation, ground_y, ground_info = ransac_ground(ground_points)

    metric_scale = metric_scale_metadata(args.meters_per_unit)
    # Identity preserves relative tracker coordinates; it is not a calibration.
    meters_per_unit = float(args.meters_per_unit) if args.meters_per_unit is not None else 1.0
    scale_status = "metric-calibrated" if metric_scale["status"] == "calibrated" else "relative-unscaled"

    def world_to_env(points: np.ndarray) -> np.ndarray:
        preliminary = transform_points(points, base_rotation, base_translation)
        aligned = preliminary @ gravity_rotation.T
        aligned[..., 1] -= ground_y
        return aligned * meters_per_unit

    def env_to_world(points: np.ndarray) -> np.ndarray:
        aligned = np.asarray(points, dtype=np.float64) / meters_per_unit
        aligned = aligned.copy()
        aligned[..., 1] += ground_y
        preliminary = aligned @ gravity_rotation
        return (preliminary - base_translation) @ base_rotation

    tracker_world_to_environment = np.eye(4)
    tracker_world_to_environment[:3, :3] = (
        meters_per_unit * gravity_rotation @ base_rotation
    )
    tracker_world_to_environment[:3, 3] = meters_per_unit * (
        gravity_rotation @ base_translation - np.array([0.0, ground_y, 0.0])
    )

    xyz_env = world_to_env(xyz_world)

    def valid_query_indices(t: int, indices: np.ndarray, relaxed_visibility: bool = False) -> np.ndarray:
        if len(indices) == 0:
            return indices
        born = queries[indices, 0] <= t
        threshold = 0.75 if relaxed_visibility else args.visibility
        good = born & (visibility[t, indices] > threshold) & (confidence[t, indices] > args.track_confidence)
        xy = track2d[t, indices, :2]
        good &= np.isfinite(xy).all(axis=1)
        good &= (
            (xy[:, 0] >= content_x0)
            & (xy[:, 0] < content_x1)
            & (xy[:, 1] >= content_y0)
            & (xy[:, 1] < content_y1)
        )
        good &= np.isfinite(xyz_env[t, indices]).all(axis=1)
        good &= result["coords_camera"][t, indices, 2] > 0
        return indices[good]

    # Sequential fragment association. SAM detIds are observations; track ids are
    # persistent world identities and can absorb later refresh/reappearance masks.
    def predict_fragment_center(indices: np.ndarray, target_frame: int) -> tuple[np.ndarray, float] | None:
        observations: list[tuple[float, np.ndarray]] = []
        for frame in range(target_frame):
            valid = valid_query_indices(frame, indices, relaxed_visibility=True)
            if len(valid):
                observations.append(
                    (float(timestamps[frame]), np.median(xyz_env[frame, valid], axis=0))
                )
        if not observations:
            return None
        last_time, last_center = observations[-1]
        velocity = np.zeros(3)
        if len(observations) >= 2:
            previous_time, previous_center = observations[-2]
            delta = last_time - previous_time
            if delta > 1e-6:
                velocity = (last_center - previous_center) / delta
                # Guard against a final noisy point-track jump dominating re-ID.
                scene_depth = max(float(np.linalg.norm(last_center)), 1e-3)
                max_speed = scene_depth * 0.5
                speed = float(np.linalg.norm(velocity))
                if speed > max_speed:
                    velocity *= max_speed / speed
        gap = max(0.0, float(timestamps[target_frame]) - last_time)
        return last_center + velocity * min(gap, max_gap), gap

    object_tracks: list[dict] = []
    det_to_track: dict[str, int] = {}
    max_gap = float(args.max_reid_gap_seconds)
    for frame_index in sorted(detections_by_frame):
        current = sorted(detections_by_frame[frame_index], key=lambda d: -d["score"])
        current_time = float(timestamps[frame_index])
        active_indices = [
            i for i, track in enumerate(object_tracks)
            if current_time - track["lastTime"] <= max_gap
        ]
        scores = np.zeros((len(active_indices), len(current)), dtype=np.float64)
        for row, track_index in enumerate(active_indices):
            track = object_tracks[track_index]
            previous = detection_by_id[track["detections"][-1]]
            previous_indices = det_to_queries[previous["detId"]]
            previous_valid = valid_query_indices(frame_index, previous_indices)
            for col, detection in enumerate(current):
                mask = mask_cache[detection["detId"]]
                components: list[tuple[float, float]] = []
                has_spatial_evidence = False
                hard_spatial_reject = False
                if len(previous_valid):
                    xy = track2d[frame_index, previous_valid, :2]
                    xi = np.clip(np.round(xy[:, 0]).astype(int), 0, width - 1)
                    yi = np.clip(np.round(xy[:, 1]).astype(int), 0, height - 1)
                    dilated = cv2.dilate(mask.astype(np.uint8), np.ones((7, 7), np.uint8)) > 0
                    support = float(dilated[yi, xi].mean())
                    components.append((0.45, support))
                    components.append((0.15, bbox_iou(points_bbox(xy), detection["bboxXYXY"])))

                current_indices = valid_query_indices(frame_index, det_to_queries[detection["detId"]], relaxed_visibility=True)
                previous_geo = valid_query_indices(frame_index, previous_indices, relaxed_visibility=True)
                if len(current_indices) and len(previous_geo):
                    a = np.median(xyz_env[frame_index, previous_geo], axis=0)
                    b = np.median(xyz_env[frame_index, current_indices], axis=0)
                    distance = float(np.linalg.norm(a - b))
                    scene_scale = max(float(np.median(np.linalg.norm(xyz_env[frame_index, current_indices], axis=1))) * 0.12, 0.08)
                    hard_spatial_reject = distance > 4.0 * scene_scale
                    components.append((0.20, math.exp(-distance / scene_scale)))
                    has_spatial_evidence = True
                elif len(current_indices):
                    historical = predict_fragment_center(previous_indices, frame_index)
                    if historical is not None:
                        predicted_center, observed_gap = historical
                        b = np.median(xyz_env[frame_index, current_indices], axis=0)
                        scene_depth = max(
                            float(np.linalg.norm(predicted_center)),
                            float(np.linalg.norm(b)),
                            1e-3,
                        )
                        allowance = max(0.08 * scene_depth * (1.0 + observed_gap), 0.10)
                        distance = float(np.linalg.norm(predicted_center - b))
                        hard_spatial_reject = distance > 2.5 * allowance
                        components.append((0.30, math.exp(-distance / allowance)))
                        has_spatial_evidence = True

                # Generic automatic masks have no class evidence. Appearance is
                # therefore allowed to refine a spatially plausible match, never
                # to merge two arbitrary same-colour objects by itself.
                if hard_spatial_reject or not has_spatial_evidence:
                    continue
                appearance = float(np.clip(np.dot(hist_cache[previous["detId"]], hist_cache[detection["detId"]]), 0, 1))
                components.append((0.25, appearance))

                same_label = previous["label"] == detection["label"]
                generic = previous["label"].lower() in GENERIC_LABELS or detection["label"].lower() in GENERIC_LABELS
                if not generic:
                    components.append((0.05, 1.0 if same_label else 0.0))
                weight = sum(w for w, _ in components)
                gap_decay = math.exp(-max(0.0, current_time - track["lastTime"]) / max(max_gap, 1e-6) * 0.20)
                scores[row, col] = sum(w * value for w, value in components) / weight * gap_decay

        matched_current: set[int] = set()
        if scores.size:
            # Rejecting sub-threshold pairs only after rectangular Hungarian can
            # sacrifice a valid edge in favour of two invalid ones.
            for row, col in thresholded_assignment(scores, args.association):
                track_index = active_indices[int(row)]
                detection = current[int(col)]
                object_tracks[track_index]["detections"].append(detection["detId"])
                object_tracks[track_index]["lastFrame"] = frame_index
                object_tracks[track_index]["lastTime"] = current_time
                det_to_track[detection["detId"]] = track_index
                matched_current.add(int(col))
        for col, detection in enumerate(current):
            if col in matched_current:
                continue
            track_index = len(object_tracks)
            object_tracks.append(
                {
                    "detections": [detection["detId"]],
                    "birthFrame": frame_index,
                    "lastFrame": frame_index,
                    "lastTime": current_time,
                }
            )
            det_to_track[detection["detId"]] = track_index

    # Dense observed geometry from masks and refined per-frame point maps.
    def dense_mask_points(
        detection: dict, maximum: int | None = 3500, *, erode: bool = True,
    ) -> np.ndarray:
        t = int(detection["frameIndex"])
        mask = mask_cache[detection["detId"]].copy()
        if erode:
            mask = cv2.erode(mask.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0
        mask &= valid_content_mask
        valid = mask & (depth_confidence[t] > 0.5) & np.isfinite(point_map[t]).all(axis=0) & (point_map[t, 2] > 0)
        ys, xs = np.where(valid)
        if maximum is not None and len(xs) > maximum:
            local_rng = np.random.default_rng(int(hashlib.sha1(detection["detId"].encode()).hexdigest()[:8], 16))
            chosen = local_rng.choice(len(xs), maximum, replace=False)
            ys, xs = ys[chosen], xs[chosen]
        if len(xs) == 0:
            return np.empty((0, 3))
        camera_points = point_map[t].transpose(1, 2, 0)[ys, xs]
        world_points = transform_points(camera_points, c2w[t, :3, :3], c2w[t, :3, 3])
        return world_to_env(world_points)

    camera_transforms: list[np.ndarray] = []
    for t in range(frame_count):
        preliminary = np.eye(4)
        preliminary[:3, :3] = basis @ first_w2c[:3, :3] @ c2w[t, :3, :3] @ basis
        preliminary[:3, 3] = basis @ (
            first_w2c[:3, :3] @ c2w[t, :3, 3] + first_w2c[:3, 3]
        )
        transform = np.eye(4)
        transform[:3, :3] = gravity_rotation @ preliminary[:3, :3]
        transform[:3, 3] = gravity_rotation @ preliminary[:3, 3]
        transform[1, 3] -= ground_y
        transform[:3, 3] *= meters_per_unit
        camera_transforms.append(transform)

    def predicted_in_view(center_env: np.ndarray, t: int) -> bool:
        world = env_to_world(np.asarray(center_env)[None])[0]
        camera = w2c[t, :3, :3] @ world + w2c[t, :3, 3]
        if not np.isfinite(camera).all() or camera[2] <= 1e-6:
            return False
        pixel = intrinsics[t] @ camera
        x, y = pixel[0] / pixel[2], pixel[1] / pixel[2]
        return bool(content_x0 <= x < content_x1 and content_y0 <= y < content_y1)

    exported_tracks: list[dict] = []
    for track_index, object_track in enumerate(object_tracks):
        fragment_ids = object_track["detections"]
        object_query_indices = np.unique(
            np.concatenate([det_to_queries[det_id] for det_id in fragment_ids])
        )
        birth_frame = min(int(detection_by_id[d]["frameIndex"]) for d in fragment_ids)
        detections_at_frame: dict[int, list[dict]] = defaultdict(list)
        for det_id in fragment_ids:
            detections_at_frame[int(detection_by_id[det_id]["frameIndex"])].append(detection_by_id[det_id])

        measurements: list[dict | None] = [None] * frame_count
        observed_world_bounds: list[tuple[np.ndarray, np.ndarray] | None] = [None] * frame_count
        previous_yaw: float | None = None
        for t in range(birth_frame, frame_count):
            dense_sets = [dense_mask_points(d) for d in detections_at_frame.get(t, [])]
            dense_sets = [p for p in dense_sets if len(p) >= 12]
            valid_indices = valid_query_indices(t, object_query_indices)
            tracked_points = xyz_env[t, valid_indices] if len(valid_indices) else np.empty((0, 3))
            points = np.concatenate(dense_sets, axis=0) if dense_sets else tracked_points
            # Collision geometry includes every valid full-mask depth point and
            # visible confident track: no erosion, subsampling or quantiles.
            collision_sets = [dense_mask_points(d, maximum=None, erode=False)
                              for d in detections_at_frame.get(t, [])]
            collision_sets.append(tracked_points)
            collision_points = np.concatenate(collision_sets, axis=0)
            collision_points = collision_points[np.isfinite(collision_points).all(axis=1)]
            if len(collision_points):
                observed_world_bounds[t] = (collision_points.min(axis=0), collision_points.max(axis=0))
            estimate = fit_yaw_box(points, previous_yaw)
            if estimate is None and len(collision_points):
                estimate = fit_yaw_box(collision_points, previous_yaw)
            if estimate is None:
                continue
            previous_yaw = float(estimate["yaw"])
            if len(valid_indices):
                measurement_confidence = float(
                    np.clip(np.mean(confidence[t, valid_indices]) * np.mean(np.minimum(visibility[t, valid_indices], 1.0)), 0, 1)
                )
            else:
                measurement_confidence = float(np.mean([d["score"] for d in detections_at_frame[t]]))
            estimate["confidence"] = measurement_confidence
            estimate["denseMaskGeometry"] = bool(dense_sets)
            measurements[t] = estimate

        # Size is an object prior, not a per-frame sparse-point extent. Prefer
        # dense SAM-mask/depth geometry; otherwise fall back to tracked points.
        measured_sizes = [
            m["size"]
            for m in measurements
            if m is not None and m["denseMaskGeometry"]
        ]
        if not measured_sizes:
            measured_sizes = [m["size"] for m in measurements if m is not None]
        if not measured_sizes:
            continue
        reference_size = np.median(np.stack(measured_sizes), axis=0)
        reference_size = np.maximum(reference_size, max(float(reference_size.max()) * 0.04, 1e-3))

        state_filter = ConstantVelocityFilter()
        samples: list[dict] = []
        last_time = float(timestamps[birth_frame])
        last_observed_time: float | None = None
        last_observed_confidence = 0.5
        filtered_yaw: float | None = None
        yaw_rate = 0.0
        was_missing = False
        for t in range(birth_frame, frame_count):
            current_time = float(timestamps[t])
            dt = max(0.0, current_time - last_time)
            measurement = measurements[t]
            # Never extrapolate beyond the configured horizon.  If a later
            # measurement reacquires the object we still predict the full gap
            # before updating, but missing-only samples freeze at the horizon.
            prediction_dt = dt
            if measurement is None and last_observed_time is not None:
                elapsed_before = max(0.0, last_time - last_observed_time)
                prediction_dt = min(
                    dt, max(0.0, args.max_prediction_seconds - elapsed_before)
                )
            state_filter.predict(prediction_dt)
            reacquired = False
            if measurement is not None:
                variance = max(0.01, (1.0 - measurement["confidence"]) * 0.25)
                state_filter.update(measurement["center"], variance)
                if filtered_yaw is None:
                    filtered_yaw = float(measurement["yaw"])
                else:
                    candidates = [measurement["yaw"] + k * math.pi for k in range(-3, 4)]
                    target_yaw = min(candidates, key=lambda value: abs(value - filtered_yaw))
                    new_rate = (target_yaw - filtered_yaw) / max(dt, 1e-6)
                    yaw_rate = 0.75 * yaw_rate + 0.25 * new_rate
                    filtered_yaw = 0.65 * filtered_yaw + 0.35 * target_yaw
                reacquired = was_missing
                was_missing = False
                last_observed_time = current_time
                last_observed_confidence = float(measurement["confidence"])
                state = "observed"
                pose_source = "fused" if measurement["denseMaskGeometry"] else "tracker"
            else:
                if state_filter.x is None:
                    last_time = current_time
                    continue
                missing = current_time - (last_observed_time if last_observed_time is not None else current_time)
                was_missing = True
                if filtered_yaw is None:
                    filtered_yaw = 0.0
                filtered_yaw += yaw_rate * prediction_dt
                if missing > args.max_prediction_seconds:
                    state_filter.x[3:] = 0.0
                    held_dt = max(0.0, dt - prediction_dt)
                    state_filter.p[:3, :3] += np.eye(3) * state_filter.q * held_dt * held_dt
                    state = "lost"
                    pose_source = "held"
                else:
                    state = "occluded" if predicted_in_view(state_filter.x[:3], t) else "out_of_view"
                    pose_source = "predicted"

            center = state_filter.x[:3].copy()
            velocity = state_filter.x[3:].copy()
            missing_duration = 0.0 if last_observed_time is None else max(0.0, current_time - last_observed_time)
            overall_confidence = (
                float(measurement["confidence"])
                if measurement is not None
                else last_observed_confidence * math.exp(-missing_duration / 2.0)
            )
            covariance_std = np.sqrt(np.maximum(np.diag(state_filter.p)[:3], 0))
            quaternion = Rotation.from_euler("y", filtered_yaw or 0.0).as_quat()
            samples.append(
                {
                    "sampledFrame": t,
                    "sourceFrame": int(source_frames[t]),
                    "t": round(current_time, 6),
                    "center": rounded(center),
                    "size": rounded(reference_size),
                    "rotation": rounded(quaternion, 7),
                    "velocity": rounded(velocity),
                    "state": state,
                    "poseSource": pose_source,
                    "reacquired": bool(reacquired),
                    "confidence": {
                        "overall": round(float(np.clip(overall_confidence, 0, 1)), 6),
                        "pointTracking": round(float(measurement["confidence"]), 6) if measurement is not None else 0.0,
                        "segmentation": round(float(np.mean([d["score"] for d in detections_at_frame.get(t, [])])), 6) if detections_at_frame.get(t) else 0.0,
                    },
                    "uncertainty": {"centerStd": rounded(covariance_std)},
                }
            )
            last_time = current_time

        if not samples:
            continue
        # Bounds are independent of the robust visual size prior. Re-express
        # each full observed AABB in the rounded, filtered pose actually exported,
        # then take their union in object-local coordinates. This also encloses
        # geometry displaced by Kalman/yaw filtering and serialization rounding.
        collision_margin = max(1e-4, 0.01 * float(np.linalg.norm(reference_size)))
        collision_low, collision_high = np.full(3, -0.5), np.full(3, 0.5)
        observed_bound_frames = 0
        for sample in samples:
            observed = observed_world_bounds[sample["sampledFrame"]]
            if observed is None:
                continue
            low, high = collision_bounds_in_pose(
                observed, np.asarray(sample["center"]),
                Rotation.from_quat(sample["rotation"]).as_matrix(),
                np.asarray(sample["size"]), collision_margin,
            )
            collision_low = np.minimum(collision_low, low)
            collision_high = np.maximum(collision_high, high)
            observed_bound_frames += 1
        # Outward rounding must not trim any part of the conservative box.
        collision_bounds = {
            "type": "box",
            "min": (np.floor(collision_low * 1e6) / 1e6).tolist(),
            "max": (np.ceil(collision_high * 1e6) / 1e6).tolist(),
        }
        labels = [detection_by_id[d]["label"] for d in fragment_ids]
        specific_labels = [label for label in labels if label.lower() not in GENERIC_LABELS]
        label = Counter(specific_labels or labels).most_common(1)[0][0]
        class_scores = [detection_by_id[d]["score"] for d in fragment_ids if detection_by_id[d]["label"] == label]
        track_id = f"trk-{len(exported_tracks) + 1:04d}"
        pose_birth_frame = int(samples[0]["sampledFrame"])
        exported_tracks.append(
            {
                "trackId": track_id,
                "entityId": f"{track_id}-entity",
                "label": label,
                "color": PALETTE[len(exported_tracks) % len(PALETTE)],
                "class": {"label": label, "confidence": round(float(np.mean(class_scores or [0.0])), 6)},
                "lifecycle": {
                    "birthFrame": int(source_frames[pose_birth_frame]),
                    "birthSampledFrame": pose_birth_frame,
                    "birthT": round(float(timestamps[pose_birth_frame]), 6),
                    "firstDetectionFrame": int(source_frames[birth_frame]),
                    "firstDetectionSampledFrame": birth_frame,
                    "firstDetectionT": round(float(timestamps[birth_frame]), 6),
                    "lastObservedFrame": max(
                        sample["sourceFrame"] for sample in samples if sample["state"] == "observed"
                    ),
                    "termination": "end-of-video",
                },
                "lineage": {
                    "sam3FragmentIds": fragment_ids,
                    "worldIdentityMethod": "point-support+appearance+3d+class",
                },
                "stateTimeline": compress_timeline(samples),
                "collisionBounds": collision_bounds,
                "collisionBoundsEvidence": {
                    "method": "union-full-valid-observations-in-exported-pose",
                    "observedFrames": observed_bound_frames,
                    "marginOutputUnits": collision_margin,
                    "marginRule": "max(1e-4 output units, 1% robust visual diagonal)",
                    "validity": "mask depth confidence > 0.5 and positive finite depth; or visible confident tracks",
                    "scope": "observed geometry at sampled times; hidden surfaces and prediction errors are not certified",
                },
                "samples": samples,
            }
        )

    if not exported_tracks:
        raise RuntimeError("No 3D object track could be fitted from the valid SAM/SpaTracker points.")

    # Environment-v1 with typed metricScale provenance; sidecar keeps audit evidence.
    all_centers = np.concatenate(
        [np.asarray([s["center"] for s in track["samples"]]) for track in exported_tracks], axis=0
    )
    target = np.median(all_centers, axis=0)
    spans = np.ptp(all_centers, axis=0)
    radial_extent = max(
        float(
            np.max(
                np.abs(np.asarray(sample["center"])[[0, 2]])
                + 0.5 * np.asarray(sample["size"])[[0, 2]]
            )
        )
        for track in exported_tracks
        for sample in track["samples"]
    )
    world_size = max(
        10.0,
        float(np.max(spans)) * 2.5 + 4.0,
        radial_extent * 2.4,
    )
    camera_distance = max(8.0, world_size * 0.85)
    overview_position = target + np.array([camera_distance, camera_distance * 0.65, camera_distance])

    entities = []
    anchors = []
    native_fps = float(metadata["sourceFps"])
    for track in exported_tracks:
        samples = track["samples"]
        birth_t = float(track["lifecycle"]["birthT"])
        position_keyframes = [{"t": s["t"], "value": s["center"]} for s in samples]
        rotation_keyframes = [{"t": s["t"], "value": s["rotation"]} for s in samples]
        scale_keyframes = []
        if birth_t > 0:
            scale_keyframes.append({"t": 0.0, "value": [0.0, 0.0, 0.0]})
            pre_birth_t = round(max(0.0, birth_t - 1.0 / max(native_fps, 1e-6)), 6)
            if pre_birth_t > 0:
                scale_keyframes.append({"t": pre_birth_t, "value": [0.0, 0.0, 0.0]})
        scale_keyframes.append({"t": round(birth_t, 6), "value": samples[0]["size"]})
        entity = {
            "id": track["entityId"],
            "label": f"{track['label']} {track['trackId']}",
            "transform": {
                "space": "world",
                "position": {"interpolation": "linear", "extrapolation": "hold", "keyframes": position_keyframes},
                "rotation": {"interpolation": "slerp", "extrapolation": "hold", "keyframes": rotation_keyframes},
                "scale": {"interpolation": "step", "extrapolation": "hold", "keyframes": scale_keyframes},
            },
            "visual": {
                "type": "primitive", "shape": "box",
                "params": {"size": [1.0, 1.0, 1.0]}, "color": track["color"],
            },
            "bounds": track["collisionBounds"],
        }
        entities.append(entity)
        anchors.append(
            {
                "id": track["trackId"], "entityId": track["entityId"],
                "label": entity["label"], "localAnchor": [0.0, 0.0, 0.0],
                "localBounds": {"type": "box", "min": [-0.5, -0.5, -0.5], "max": [0.5, 0.5, 0.5]},
            }
        )

    environment = {
        "schemaVersion": "1.0",
        "kind": "environment",
        "id": args.environment_id,
        "promptExampleId": args.environment_id,
        "prompt": (
            "Object-centric 4D environment reconstructed from video"
            if scale_status == "metric-calibrated"
            else "RELATIVE-SCALE object-centric 4D environment; calibrate before metric use"
        ),
        "clock": {
            "durationSeconds": round(float(metadata["durationSeconds"]), 6),
            "timeDomain": "playback",
            "fpsHint": round(native_fps, 6),
        },
        "coordinates": {
            "handedness": "right", "upAxis": "+Y", "cameraForwardAxis": "-Z",
            "lengthUnit": "meter", "rotationOrder": "quaternion-xyzw",
            "metricScale": metric_scale,
        },
        "evaluation": {"distanceMetric": "boundsSurface", "epsilon": 0.001},
        "world": {
            "profile": "studio",
            "background": "#6f747b",
            "overviewCamera": {"position": rounded(overview_position), "target": rounded(target)},
            "ground": {"y": 0.0, "size": [round(world_size, 4), round(world_size, 4)], "color": "#777777", "surface": "concrete"},
            "grid": {"size": round(world_size, 4), "divisions": 20},
        },
        "entities": entities,
        "semanticAnchors": anchors,
    }

    camera_samples = []
    for t, transform in enumerate(camera_transforms):
        camera_samples.append(
            {
                "sampledFrame": t,
                "sourceFrame": int(source_frames[t]),
                "t": round(float(timestamps[t]), 6),
                "position": rounded(transform[:3, 3]),
                "rotation": rounded(Rotation.from_matrix(transform[:3, :3]).as_quat(), 7),
                "intrinsics": {
                    "fx": round(float(intrinsics[t, 0, 0]), 6),
                    "fy": round(float(intrinsics[t, 1, 1]), 6),
                    "cx": round(float(intrinsics[t, 0, 2]), 6),
                    "cy": round(float(intrinsics[t, 1, 2]), 6),
                    "imageSize": [width, height],
                },
            }
        )

    sidecar = {
        "schemaVersion": "1.0",
        "kind": "environment-tracking",
        "environmentId": args.environment_id,
        "source": {
            "videoName": metadata["sourceVideoName"],
            "sourceFps": native_fps,
            "sourceFrameCount": int(metadata["sourceFrameCount"]),
            "timestampMode": metadata.get("timestampMode", "constant-fps-legacy"),
            "sampledSourceFrameIndices": source_frames.tolist(),
            "sampledTimestampsSeconds": rounded(timestamps),
            "letterbox": metadata["letterbox"],
        },
        "scale": {
            "status": scale_status,
            **({"metersPerTrackerUnit": float(args.meters_per_unit)} if args.meters_per_unit is not None else {}),
            "appliedCoordinateFactor": meters_per_unit,
            "warning": None if metric_scale["status"] == "calibrated" else "Monocular scale is relative; provide METERS_PER_TRACKER_UNIT before metric use.",
        },
        "coordinates": {
            "trackerConvention": "+X right, +Y down, +Z forward",
            "environmentConvention": "+X right, +Y up, -Z forward",
            "trackerWorldToEnvironment": tracker_world_to_environment.round(9).tolist(),
            "gravityAlignment": ground_info,
        },
        "sources": {
            "sam3": {
                "revision": metadata["revisions"]["sam3"],
                "resolvedModelRevision": discovery_payload.get("resolvedModelRevision"),
                "mode": "sparse-image-automatic-mask-generation",
                "settings": discovery_payload.get("automaticMaskGeneration", {}),
            },
            "querySeeding": {
                "pointsPerObject": int(query_payload["points_per_object"].item())
                if "points_per_object" in query_payload.files
                else None,
                "maximumQueries": int(query_payload["maximum_queries"].item())
                if "maximum_queries" in query_payload.files
                else None,
                "availableMaskFragments": int(
                    query_payload["available_detection_count"].item()
                )
                if "available_detection_count" in query_payload.files
                else None,
                "selectedMaskFragments": int(
                    query_payload["selected_detection_count"].item()
                )
                if "selected_detection_count" in query_payload.files
                else None,
                "depthValidQueries": int(len(kept)),
            },
            "spatialTrackerV2": {
                "revision": metadata["revisions"]["spatracker"],
                "frontCheckpoint": "Yuxihenry/SpatialTrackerV2_Front",
                "frontCheckpointRevision": str(result["front_model_revision"].item())
                if "front_model_revision" in result.files
                else None,
                "trackerCheckpoint": "Yuxihenry/SpatialTrackerV2-Online",
                "trackerCheckpointRevision": str(result["tracker_model_revision"].item())
                if "tracker_model_revision" in result.files
                else None,
                "localCompatibilityPatches": [
                    "T4 math/memory-efficient SDPA instead of flash-only attention",
                    "preserve query order/count after external depth-confidence validation",
                ],
                "randomSeed": int(result["random_seed"].item())
                if "random_seed" in result.files
                else None,
                "patchedFileSha256": {
                    "attention": str(result["patched_attention_sha256"].item())
                    if "patched_attention_sha256" in result.files
                    else None,
                    "queryPath": str(result["patched_query_path_sha256"].item())
                    if "patched_query_path_sha256" in result.files
                    else None,
                },
            },
            "fusion": {"method": "fragment-association+robust-yaw-obb+constant-velocity-kalman", "version": "1"},
        },
        "cameraTrajectory": {"id": "capture-camera", "poseConvention": "cameraToWorld", "samples": camera_samples},
        "tracks": [{k: v for k, v in track.items() if k != "color"} for track in exported_tracks],
    }

    environment_path = args.output / "scene.environment.json"
    sidecar_path = args.output / "scene.tracking.json"
    environment_path.write_text(
        json.dumps(environment, indent=2, ensure_ascii=False, allow_nan=False) + "\n"
    )
    sidecar_path.write_text(
        json.dumps(sidecar, indent=2, ensure_ascii=False, allow_nan=False) + "\n"
    )

    # 3D trajectory preview.
    figure = plt.figure(figsize=(10, 8))
    axis = figure.add_subplot(111, projection="3d")
    for track in exported_tracks:
        centers = np.asarray([s["center"] for s in track["samples"]])
        axis.plot(centers[:, 0], centers[:, 2], centers[:, 1], color=track["color"], label=f"{track['trackId']} {track['label']}")
        axis.scatter(centers[0, 0], centers[0, 2], centers[0, 1], color=track["color"], marker="o")
    axis.set_xlabel("X")
    axis.set_ylabel("Z")
    axis.set_zlabel("Y up")
    axis.set_title("Recovered object centers (sampled times)")
    axis.legend(fontsize=7, loc="best")
    figure.tight_layout()
    figure.savefig(args.output / "trajectory_preview.png", dpi=160)
    plt.close(figure)

    # Lightweight 2D audit video based on valid tracked points.
    writer = cv2.VideoWriter(
        str(args.output / "tracking_preview.mp4"),
        cv2.VideoWriter_fourcc(*"mp4v"), 4.0, (width, height),
    )
    for t in range(frame_count):
        frame = cv2.imread(str(args.frames / f"frame_{t:04d}.jpg"))
        for track in exported_tracks:
            fragment_indices = np.unique(
                np.concatenate([det_to_queries[d] for d in track["lineage"]["sam3FragmentIds"]])
            )
            valid = valid_query_indices(t, fragment_indices)
            if len(valid) < 2:
                continue
            xy = track2d[t, valid, :2]
            x0, y0, x1, y1 = [int(round(v)) for v in points_bbox(xy)]
            color_rgb = tuple(int(track["color"][i : i + 2], 16) for i in (1, 3, 5))
            color_bgr = color_rgb[::-1]
            cv2.rectangle(frame, (x0, y0), (x1, y1), color_bgr, 2)
            cv2.putText(frame, f"{track['trackId']} {track['label']}", (x0, max(16, y0 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color_bgr, 1, cv2.LINE_AA)
        writer.write(frame)
    writer.release()

    print(f"Exported {len(exported_tracks)} persistent 3D object tracks")
    print(environment_path)
    print(sidecar_path)
    if scale_status != "metric-calibrated":
        print("WARNING: monocular scale is relative; strict env lengthUnit remains 'meter' only for schema compatibility.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/4d_env/scripts/validate_outputs.py
#!/usr/bin/env python3
"""Strict structural and numeric checks for the inferred environment-v1 contract."""

from __future__ import annotations

import argparse
import json
import math
from pathlib import Path


ROOT_KEYS = {
    "schemaVersion", "kind", "id", "promptExampleId", "prompt", "clock",
    "coordinates", "evaluation", "world", "entities", "semanticAnchors",
}
ENTITY_KEYS = {"id", "label", "transform", "visual", "bounds"}
TRANSFORM_KEYS = {"space", "position", "rotation", "scale"}
ANCHOR_KEYS = {"id", "entityId", "label", "localAnchor", "localBounds"}


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--environment", type=Path, required=True)
    parser.add_argument("--tracking", type=Path, required=True)
    return parser.parse_args()


def walk_finite(value, path="root") -> None:
    if isinstance(value, float) and not math.isfinite(value):
        raise AssertionError(f"Non-finite number at {path}: {value}")
    if isinstance(value, dict):
        for key, child in value.items():
            walk_finite(child, f"{path}.{key}")
    elif isinstance(value, list):
        for index, child in enumerate(value):
            walk_finite(child, f"{path}[{index}]")


def validate_curve(curve: dict, width: int, duration: float, *, quaternion: bool = False) -> None:
    assert set(curve) == {"interpolation", "extrapolation", "keyframes"}
    assert curve["extrapolation"] == "hold"
    keyframes = curve["keyframes"]
    assert keyframes, "Curve has no keyframes"
    times = [float(keyframe["t"]) for keyframe in keyframes]
    assert all(b > a for a, b in zip(times, times[1:])), (
        f"Keyframe times must be strictly increasing: {times}"
    )
    assert min(times) >= -1e-6 and max(times) <= duration + 1e-5
    for keyframe in keyframes:
        value = keyframe["value"]
        assert isinstance(value, list) and len(value) == width
        if quaternion:
            norm = math.sqrt(sum(float(component) ** 2 for component in value))
            assert abs(norm - 1.0) < 2e-3, f"Quaternion is not unit length: {norm}"


def main() -> None:
    args = parse_args()
    environment = json.loads(args.environment.read_text())
    tracking = json.loads(args.tracking.read_text())
    walk_finite(environment)
    walk_finite(tracking)

    assert set(environment) == ROOT_KEYS, f"Unexpected/missing root keys: {set(environment) ^ ROOT_KEYS}"
    assert environment["schemaVersion"] == "1.0"
    assert environment["kind"] == "environment"
    assert set(environment["clock"]) == {"durationSeconds", "timeDomain", "fpsHint"}
    assert environment["clock"]["timeDomain"] == "playback"
    assert float(environment["clock"]["durationSeconds"]) >= 0
    assert float(environment["clock"]["fpsHint"]) > 0
    coordinates = environment["coordinates"]
    assert {key: value for key, value in coordinates.items() if key != "metricScale"} == {
        "handedness": "right", "upAxis": "+Y", "cameraForwardAxis": "-Z",
        "lengthUnit": "meter", "rotationOrder": "quaternion-xyzw",
    }
    metric_scale = coordinates["metricScale"]
    assert metric_scale["status"] in {"relative", "calibrated"}
    if metric_scale["status"] == "calibrated":
        assert set(metric_scale) == {"status", "metersPerUnit"}
        assert math.isfinite(metric_scale["metersPerUnit"]) and metric_scale["metersPerUnit"] > 0
        assert tracking["scale"]["status"] == "metric-calibrated"
        assert metric_scale["metersPerUnit"] == 1.0
        assert tracking["scale"]["metersPerTrackerUnit"] > 0
        assert tracking["scale"]["appliedCoordinateFactor"] == tracking["scale"]["metersPerTrackerUnit"]
    else:
        assert metric_scale == {"status": "relative"}
        assert tracking["scale"]["status"] == "relative-unscaled"
        assert "metersPerTrackerUnit" not in tracking["scale"]
    duration = float(environment["clock"]["durationSeconds"])
    assert set(environment["evaluation"]) == {"distanceMetric", "epsilon"}
    assert environment["evaluation"]["distanceMetric"] == "boundsSurface"
    assert float(environment["evaluation"]["epsilon"]) > 0
    assert set(environment["world"]) == {
        "profile", "background", "overviewCamera", "ground", "grid"
    }
    entity_ids = set()
    for entity in environment["entities"]:
        assert set(entity) == ENTITY_KEYS
        assert entity["id"] not in entity_ids
        entity_ids.add(entity["id"])
        transform = entity["transform"]
        assert set(transform) == TRANSFORM_KEYS and transform["space"] == "world"
        validate_curve(transform["position"], 3, duration)
        validate_curve(transform["rotation"], 4, duration, quaternion=True)
        validate_curve(transform["scale"], 3, duration)
        assert transform["position"]["interpolation"] == "linear"
        assert transform["rotation"]["interpolation"] == "slerp"
        assert transform["scale"]["interpolation"] == "step"
        positive_scales = [k["value"] for k in transform["scale"]["keyframes"] if max(k["value"]) > 0]
        assert positive_scales and all(min(scale) > 0 for scale in positive_scales)
        bounds = entity["bounds"]
        assert set(bounds) == {"type", "min", "max"} and bounds["type"] == "box"
        assert len(bounds["min"]) == len(bounds["max"]) == 3
        assert all(low <= -0.5 and high >= 0.5 for low, high in zip(bounds["min"], bounds["max"]))
        assert entity["visual"]["type"] == "primitive"
        assert entity["visual"]["shape"] == "box"
        assert entity["visual"]["params"] == {"size": [1.0, 1.0, 1.0]}

    anchor_ids = set()
    for anchor in environment["semanticAnchors"]:
        assert set(anchor) == ANCHOR_KEYS and anchor["entityId"] in entity_ids
        assert anchor["id"] not in anchor_ids
        anchor_ids.add(anchor["id"])
        assert anchor["localAnchor"] == [0.0, 0.0, 0.0]
        assert anchor["localBounds"] == {
            "type": "box", "min": [-0.5, -0.5, -0.5], "max": [0.5, 0.5, 0.5]
        }
    assert tracking["environmentId"] == environment["id"]
    sidecar_entity_ids = {track["entityId"] for track in tracking["tracks"]}
    assert sidecar_entity_ids == entity_ids
    allowed_states = {"observed", "occluded", "out_of_view", "lost"}
    for track in tracking["tracks"]:
        assert track["samples"]
        sample_times = [float(sample["t"]) for sample in track["samples"]]
        assert all(b > a for a, b in zip(sample_times, sample_times[1:]))
        assert all(sample["state"] in allowed_states for sample in track["samples"])
        assert track["lifecycle"]["birthT"] == track["samples"][0]["t"]

    print(f"PASS: strict environment-v1 structure, {len(entity_ids)} entities")
    print("PASS: finite values, ordered timestamps, positive dimensions, unit quaternions")
    if tracking["scale"]["status"] != "metric-calibrated":
        assert environment["prompt"].startswith("RELATIVE-SCALE"), (
            "An uncalibrated strict environment must carry an in-file scale warning"
        )
        print("WARNING: scale is relative, not physically calibrated meters")


if __name__ == "__main__":
    main()


In [ ]:
fusion_command = [
    sys.executable, WORK_ROOT / "scripts/fuse_and_export.py",
    "--metadata", VIDEO_METADATA,
    "--detections", SAM_DETECTIONS,
    "--queries", QUERY_PATH,
    "--tracks", TRACK_PATH,
    "--frames", FRAME_ROOT / "frames",
    "--output", OUTPUT_ROOT,
    "--environment-id", ENVIRONMENT_ID,
    "--visibility", VISIBILITY_THRESHOLD,
    "--track-confidence", TRACK_CONFIDENCE,
    "--association", ASSOCIATION_THRESHOLD,
    "--max-reid-gap-seconds", MAX_REID_GAP_SECONDS,
    "--max-prediction-seconds", MAX_PREDICTION_SECONDS,
]
shutil.rmtree(OUTPUT_ROOT, ignore_errors=True)
(WORK_ROOT / "environment_bundle.zip").unlink(missing_ok=True)
if METERS_PER_TRACKER_UNIT is not None:
    fusion_command.extend(["--meters-per-unit", METERS_PER_TRACKER_UNIT])
run(fusion_command)

ENV_PATH = OUTPUT_ROOT / "scene.environment.json"
SIDECAR_PATH = OUTPUT_ROOT / "scene.tracking.json"
run([
    sys.executable, WORK_ROOT / "scripts/validate_outputs.py",
    "--environment", ENV_PATH,
    "--tracking", SIDECAR_PATH,
])

# Keep raw evidence in the downloadable bundle.
shutil.copy2(TRACK_PATH, OUTPUT_ROOT / "raw_spatial_tracks.npz")
shutil.copy2(QUERY_PATH, OUTPUT_ROOT / "object_queries.npz")
shutil.copy2(SAM_DETECTIONS, OUTPUT_ROOT / "sam3_detections.json")
shutil.copytree(SAM_ROOT / "masks", OUTPUT_ROOT / "masks", dirs_exist_ok=True)
shutil.copytree(FRAME_ROOT / "frames", OUTPUT_ROOT / "frames", dirs_exist_ok=True)
shutil.copy2(VIDEO_METADATA, OUTPUT_ROOT / "video_metadata.json")
bundled_scripts = OUTPUT_ROOT / "pipeline_scripts"
bundled_scripts.mkdir(exist_ok=True)
for script_path in (WORK_ROOT / "scripts").glob("*.py"):
    shutil.copy2(script_path, bundled_scripts / script_path.name)
(OUTPUT_ROOT / "sam_environment.freeze.txt").write_text(
    subprocess.check_output(
        [UV, "pip", "freeze", "--python", SAM_PY], text=True
    )
)
(OUTPUT_ROOT / "spatracker_environment.freeze.txt").write_text(
    subprocess.check_output(
        [UV, "pip", "freeze", "--python", SPA_PY], text=True
    )
)


## 8. کنترل هندسی و preview

علاوه بر schema validation، reprojectionِ نقطه‌های سه‌بعدی به track دوبعدی چک می‌شود. خطای زیاد معمولاً نشانه‌ی convention اشتباه، scene cut یا تخمین نامطمئن دوربین/depth است.


In [ ]:
import json
import numpy as np
from IPython.display import Image as DisplayImage, Video, display
from scipy.spatial.transform import Rotation

raw = np.load(OUTPUT_ROOT / "raw_spatial_tracks.npz")
query_payload = np.load(OUTPUT_ROOT / "object_queries.npz")
camera = raw["coords_camera"]
intrinsics = raw["intrinsics"]
observed_2d = raw["track2d"][..., :2]
z = camera[..., 2]
projected = np.empty_like(observed_2d)
projected[..., 0] = (
    intrinsics[:, None, 0, 0] * camera[..., 0] / np.maximum(z, 1e-6)
    + intrinsics[:, None, 0, 2]
)
projected[..., 1] = (
    intrinsics[:, None, 1, 1] * camera[..., 1] / np.maximum(z, 1e-6)
    + intrinsics[:, None, 1, 2]
)
kept_query_indices = raw["kept_query_indices"].astype(np.int64)
kept_queries = query_payload["queries"][kept_query_indices]
born = kept_queries[None, :, 0] <= np.arange(camera.shape[0])[:, None]
video_metadata = json.loads((OUTPUT_ROOT / "video_metadata.json").read_text())
content_x0, content_y0, content_x1, content_y1 = video_metadata["letterbox"]["contentRectXYXY"]
inside_content = (
    (observed_2d[..., 0] >= content_x0)
    & (observed_2d[..., 0] < content_x1)
    & (observed_2d[..., 1] >= content_y0)
    & (observed_2d[..., 1] < content_y1)
)
valid = (
    born
    & (raw["visibility"].reshape(camera.shape[:2]) > VISIBILITY_THRESHOLD)
    & (raw["track_confidence"].reshape(camera.shape[:2]) > TRACK_CONFIDENCE)
    & (z > 0)
    & inside_content
    & np.isfinite(camera).all(axis=-1)
    & np.isfinite(observed_2d).all(axis=-1)
    & np.isfinite(projected).all(axis=-1)
)
reprojection_error = np.linalg.norm(projected - observed_2d, axis=-1)[valid]
environment = json.loads(ENV_PATH.read_text())
sidecar = json.loads(SIDECAR_PATH.read_text())
if not len(reprojection_error):
    raise RuntimeError("No valid point was available for reprojection QA.")
median_error = float(np.median(reprojection_error))
p95_error = float(np.quantile(reprojection_error, 0.95))
print(f"Median 3D→2D reprojection disagreement: {median_error:.2f}px")
print(f"95th percentile: {p95_error:.2f}px")
if (
    not np.isfinite(median_error)
    or not np.isfinite(p95_error)
    or median_error > MAX_MEDIAN_REPROJECTION_ERROR_PX
    or p95_error > MAX_P95_REPROJECTION_ERROR_PX
):
    raise RuntimeError(
        "Spatial reconstruction failed reprojection QA. Try a shorter, smoother, "
        "single-shot clip or increase frame density."
    )

# Independently validate the exported tracker-world→environment matrix,
# quaternion convention, and capture-camera trajectory.
world_points = raw["coords_world"]
homogeneous = np.concatenate(
    [world_points, np.ones((*world_points.shape[:2], 1), dtype=world_points.dtype)],
    axis=-1,
)
tracker_to_env = np.asarray(
    sidecar["coordinates"]["trackerWorldToEnvironment"], dtype=np.float64
)
environment_points = np.einsum("ij,tnj->tni", tracker_to_env, homogeneous)[..., :3]
env_projected = np.empty_like(observed_2d)
cv_from_env_camera = np.diag([1.0, -1.0, -1.0])
for t, camera_sample in enumerate(sidecar["cameraTrajectory"]["samples"]):
    camera_rotation = Rotation.from_quat(camera_sample["rotation"]).as_matrix()
    camera_position = np.asarray(camera_sample["position"], dtype=np.float64)
    local_environment = (environment_points[t] - camera_position) @ camera_rotation
    local_cv = local_environment @ cv_from_env_camera
    local_z = np.maximum(local_cv[:, 2], 1e-6)
    env_projected[t, :, 0] = (
        intrinsics[t, 0, 0] * local_cv[:, 0] / local_z + intrinsics[t, 0, 2]
    )
    env_projected[t, :, 1] = (
        intrinsics[t, 1, 1] * local_cv[:, 1] / local_z + intrinsics[t, 1, 2]
    )
convention_error = np.linalg.norm(env_projected - projected, axis=-1)[valid]
convention_p95 = float(np.quantile(convention_error, 0.95))
print(f"Exported env/camera convention p95 disagreement: {convention_p95:.3f}px")
if not np.isfinite(convention_p95) or convention_p95 > 1.0:
    raise RuntimeError("Exported coordinate or quaternion convention is inconsistent.")

qa_report = {
    "status": "pass",
    "medianReprojectionErrorPx": median_error,
    "p95ReprojectionErrorPx": p95_error,
    "exportConventionP95ErrorPx": convention_p95,
    "thresholds": {
        "maxMedianReprojectionErrorPx": MAX_MEDIAN_REPROJECTION_ERROR_PX,
        "maxP95ReprojectionErrorPx": MAX_P95_REPROJECTION_ERROR_PX,
        "maxExportConventionP95ErrorPx": 1.0,
    },
}
(OUTPUT_ROOT / "qa_report.json").write_text(
    json.dumps(qa_report, indent=2, allow_nan=False) + "\n"
)
ZIP_PATH = Path(
    shutil.make_archive(str(WORK_ROOT / "environment_bundle"), "zip", OUTPUT_ROOT)
)
print("Validated bundle:", ZIP_PATH)

print(f"Entities: {len(environment['entities'])}")
print("Scale:", sidecar["scale"]["status"])
for track in sidecar["tracks"]:
    states = sorted({sample["state"] for sample in track["samples"]})
    print(track["trackId"], track["class"]["label"], "states=", states)

display(DisplayImage(filename=str(OUTPUT_ROOT / "trajectory_preview.png")))
display(Video(str(OUTPUT_ROOT / "tracking_preview.mp4"), embed=True))


In [ ]:
# Download the complete result. You can also download only scene.environment.json.
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except ImportError:
    print(ZIP_PATH)


## تفسیر درست خروجی

- `scene.environment.json` با rendererهای schema v1 سازگار می‌ماند و هیچ فیلد جدیدی به ریشه یا entity تزریق نمی‌کند.
- `overviewCamera` فقط دوربین نمایش محیط است؛ مسیر دوربینِ واقعی ویدئو در `scene.tracking.json → cameraTrajectory` است.
- `observed` یعنی measurement معتبر وجود داشته؛ `occluded/out_of_view` یعنی motion prediction؛ `lost` یعنی prediction horizon تمام شده و آخرین occupancy نگه داشته شده است.
- اگر `scale.status == "relative-unscaled"` باشد، اعداد هنوز متر فیزیکی نیستند—even though schema v1 برای سازگاری `lengthUnit: "meter"` می‌خواهد.
- هر hard cut باید جداگانه به یک environment تبدیل شود. metadata، scene-cutهای مشکوک را هشدار می‌دهد.
- اندازه‌ی hidden side یک object از RGB monocular دقیقاً مشاهده نمی‌شود؛ OBB از چند نمای دیده‌شده و prior ثابت‌اندازه تخمین زده شده است.

- `coordinates.metricScale.status` برابر `relative` یا `calibrated` است. در حالت relative هیچ ضریب متر حدس زده نمی‌شود؛ optimizer متریک این خروجی را تا کالیبراسیون نمی‌پذیرد. ضریب `metersPerUnit` در حالت calibrated برابر ۱ است: هر واحد مختصات خروجی یک متر است. ضریب اندازه‌گیری‌شدهٔ tracker به متر **قبلاً** اعمال شده و فقط در sidecar با نام `metersPerTrackerUnit` ثبت می‌شود؛ نباید دوباره ضرب شود.
- `entity.bounds` پوشش محافظه‌کارانهٔ هندسهٔ معتبر مشاهده‌شده است؛ شکل نمایشی و `semanticAnchors.localBounds` همچنان prior مقاوم OBB هستند. هندسهٔ پنهان و حرکت میان فریم‌های نمونه تضمین نشده‌اند.
